# Egyptian Arabic Word Pronunciation Experiments using Edge TTS

Generate Egyptian-style pronunciation variants for Arabic **words** from `master_dataset.csv`.
Approved number audio is reused when present; this notebook focuses on word experiments.

## Imports and Relative Paths

Import libraries and define project-relative paths (no absolute paths).

In [1]:
import asyncio
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Audio as IPA, display


def _ensure(pkg: str, import_name: str | None = None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        __import__(name)


_ensure("pandas")
_ensure("edge-tts", "edge_tts")
_ensure("nest_asyncio")

import edge_tts
import nest_asyncio

nest_asyncio.apply()

BASE_DIR = Path(".").resolve()
DATASET_PATH = BASE_DIR / "master_dataset.csv"
AUDIO_ROOT = BASE_DIR / "generated_audio_words"
DIR_WORDS = AUDIO_ROOT / "edge_tts_words"
DIR_COMPARISON = DIR_WORDS / "comparison"
DIR_COMPARISON2 = DIR_WORDS / "comparison2"
DIR_COMPARISON2_CUT = DIR_COMPARISON2 / "cut"
DIR_COMPARISON3 = DIR_WORDS / "comparison3"
DIR_COMPARISON3_CUT = DIR_COMPARISON3 / "cut"
DIR_COMPARISON4 = DIR_WORDS / "comparison4"
DIR_COMPARISON4_CUT = DIR_COMPARISON4 / "cut"
DIR_COMPARISON5 = DIR_WORDS / "comparison5"
DIR_COMPARISON5_CUT = DIR_COMPARISON5 / "cut"
DIR_COMPARISON6 = DIR_WORDS / "comparison6"
DIR_COMPARISON7 = DIR_WORDS / "comparison7"
DIR_COMPARISON7_CUT = DIR_COMPARISON7 / "cut"
DIR_CHOSEN = DIR_WORDS / "chosen"
DIR_CUT = DIR_WORDS / "cut"

# Approved number audio (reuse instead of regenerating)
APPROVED_NUMBERS_DIRS = [
    AUDIO_ROOT / "edge_tts_numbers_mapping_v1" / "chosen",
    AUDIO_ROOT / "edge_tts_numbers" / "chosen",
    AUDIO_ROOT / "edge_tts_numbers",
]

print("BASE_DIR:", BASE_DIR)
print("DATASET_PATH:", DATASET_PATH)
print("DIR_COMPARISON:", DIR_COMPARISON)
print("DIR_COMPARISON2:", DIR_COMPARISON2)
print("DIR_COMPARISON2_CUT:", DIR_COMPARISON2_CUT)
print("DIR_COMPARISON3:", DIR_COMPARISON3)
print("DIR_COMPARISON3_CUT:", DIR_COMPARISON3_CUT)
print("DIR_COMPARISON4:", DIR_COMPARISON4)
print("DIR_COMPARISON4_CUT:", DIR_COMPARISON4_CUT)
print("DIR_COMPARISON5:", DIR_COMPARISON5)
print("DIR_COMPARISON5_CUT:", DIR_COMPARISON5_CUT)
print("DIR_COMPARISON6:", DIR_COMPARISON6)
print("DIR_COMPARISON7:", DIR_COMPARISON7)
print("DIR_COMPARISON7_CUT:", DIR_COMPARISON7_CUT)

BASE_DIR: C:\Users\sondo\Desktop\Voice Model Words
DATASET_PATH: C:\Users\sondo\Desktop\Voice Model Words\master_dataset.csv
DIR_COMPARISON: C:\Users\sondo\Desktop\Voice Model Words\generated_audio_words\edge_tts_words\comparison
DIR_COMPARISON2: C:\Users\sondo\Desktop\Voice Model Words\generated_audio_words\edge_tts_words\comparison2
DIR_COMPARISON2_CUT: C:\Users\sondo\Desktop\Voice Model Words\generated_audio_words\edge_tts_words\comparison2\cut
DIR_COMPARISON3: C:\Users\sondo\Desktop\Voice Model Words\generated_audio_words\edge_tts_words\comparison3
DIR_COMPARISON3_CUT: C:\Users\sondo\Desktop\Voice Model Words\generated_audio_words\edge_tts_words\comparison3\cut
DIR_COMPARISON4: C:\Users\sondo\Desktop\Voice Model Words\generated_audio_words\edge_tts_words\comparison4
DIR_COMPARISON4_CUT: C:\Users\sondo\Desktop\Voice Model Words\generated_audio_words\edge_tts_words\comparison4\cut
DIR_COMPARISON5: C:\Users\sondo\Desktop\Voice Model Words\generated_audio_words\edge_tts_words\compariso

## Load Dataset

Load `master_dataset.csv`, inspect structure, and extract unique Arabic words from `gesture_arabic`.

In [2]:
df = pd.read_csv(DATASET_PATH)

print("shape:", df.shape)
print("\ncolumns:")
print(list(df.columns))
print("\nsample rows:")
display(df[["gesture_name", "gesture_arabic"]].drop_duplicates().head(12))

unique_words = sorted(df["gesture_arabic"].dropna().astype(str).str.strip().unique())
print("\nextracted unique words (%d):" % len(unique_words))
for w in unique_words:
    print(" ", w)

shape: (72720, 15)

columns:
['flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5', 'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'gesture_id', 'gesture_name', 'gesture_arabic', 'hand_dominance']

sample rows:


,gesture_name,gesture_arabic
0,wahed,واحد
2250,etneen,اتنين
4500,talata,تلاتة
6750,arba'a,أربعة
9000,khamsa,خمسة
11250,setta,ستة
13500,sab'a,سبعة
15750,tamanya,تمانية
18000,tis'a,تسعة
20250,ashara,عشرة



extracted unique words (35):
  آسف
  آكل
  أربعة
  أشرب
  أكل
  إزيك
  اتنين
  بحبك
  بيت
  تسعة
  تعال
  تلاتة
  تمام
  تمانية
  خمسة
  روح
  سبعة
  ستة
  سلام
  شكراً
  عايز
  عربية
  عشرة
  عفواً
  فلوس
  فين
  كلام
  كويس
  لو سمحت
  مبروك
  مش كويس
  معايا
  نام
  واحد
  يلا


## Edge TTS Setup

Select Egyptian Arabic voice (`ar-EG-SalmaNeural`, fallback `ar-EG-ShakirNeural`).

In [3]:
PREFERRED_VOICES = ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]
OPTIONAL_COMPARE_VOICE = "ar-EG-ShakirNeural"
_ALL_VOICES = None


async def _list_voices():
    global _ALL_VOICES
    if _ALL_VOICES is None:
        _ALL_VOICES = await edge_tts.list_voices()
    return _ALL_VOICES


async def select_edge_voice(preferred: list[str] | None = None) -> str:
    prefs = preferred or PREFERRED_VOICES
    voices = await _list_voices()
    by_short = {v["ShortName"]: v for v in voices}
    for pref in prefs:
        if pref in by_short:
            return pref
    ar_names = sorted(
        v["ShortName"]
        for v in voices
        if str(v.get("Locale", "")).lower().startswith("ar")
    )
    eg = [
        v["ShortName"]
        for v in voices
        if "EG" in v.get("Locale", "") or "-EG-" in v.get("ShortName", "")
    ]
    if eg:
        return eg[0]
    if ar_names:
        return ar_names[0]
    raise RuntimeError("No Arabic Edge TTS voice found")


EDGE_VOICE = asyncio.get_event_loop().run_until_complete(select_edge_voice())
print("Selected voice:", EDGE_VOICE)

if EDGE_VOICE != OPTIONAL_COMPARE_VOICE:
  try:
    alt = asyncio.get_event_loop().run_until_complete(
        select_edge_voice([OPTIONAL_COMPARE_VOICE])
    )
    print("Optional comparison voice available:", alt)
  except RuntimeError:
    print("Optional comparison voice not available:", OPTIONAL_COMPARE_VOICE)

Selected voice: ar-EG-SalmaNeural
Optional comparison voice available: ar-EG-ShakirNeural


## Create Output Folders

Create `generated_audio_words/edge_tts_words/`, `comparison/`, and `chosen/` (and `cut/` for optional trims).

In [4]:
for d in (DIR_WORDS, DIR_COMPARISON, DIR_COMPARISON2, DIR_COMPARISON2_CUT, DIR_COMPARISON3, DIR_COMPARISON3_CUT, DIR_COMPARISON4, DIR_COMPARISON4_CUT, DIR_COMPARISON5, DIR_COMPARISON5_CUT, DIR_COMPARISON6, DIR_COMPARISON7, DIR_COMPARISON7_CUT, DIR_CHOSEN, DIR_CUT):
    d.mkdir(parents=True, exist_ok=True)
    print("ready:", d.relative_to(BASE_DIR))

ready: generated_audio_words\edge_tts_words
ready: generated_audio_words\edge_tts_words\comparison
ready: generated_audio_words\edge_tts_words\comparison2
ready: generated_audio_words\edge_tts_words\comparison2\cut
ready: generated_audio_words\edge_tts_words\comparison3
ready: generated_audio_words\edge_tts_words\comparison3\cut
ready: generated_audio_words\edge_tts_words\chosen
ready: generated_audio_words\edge_tts_words\cut


## Word Pronunciation Variants

Define Egyptian-friendly TTS spellings per word (punctuation, sukoon, stretched vowels).
Numbers are listed for reference but skipped for generation — approved number MP3s are copied when found.

In [5]:
NUMBER_WORDS = {
    "واحد", "اتنين", "تلاتة", "أربعة", "خمسة", "ستة",
    "سبعة", "تمانية", "تسعة", "عشرة",
}

NUMBER_APPROVED_GLOB = {
    "واحد": ["num_01*", "num_00*", "*wahed*", "*wahid*"],
    "اتنين": ["num_02*", "*etneen*", "*itneen*", "*atneen*"],
    "تلاتة": ["num_03*", "*talata*", "*tlate*", "*thlatha*"],
    "أربعة": ["num_04*", "*arba*", "*arbaa*"],
    "خمسة": ["num_05*", "*khamsa*", "*khams*"],
    "ستة": ["num_06*", "*sitta*", "*setta*"],
    "سبعة": ["num_07*", "*sabaa*", "*sab3a*"],
    "تمانية": ["num_08*", "*tamanya*", "*tamania*"],
    "تسعة": ["num_09*", "*tisa*", "*tis3a*"],
    "عشرة": ["num_10*", "*ashara*", "*ashrah*"],
}

WORD_VARIANTS: dict[str, list[str]] = {
    "سلام": [
        "سَلَام.", "سَلاَم.", "سَلَامْ.", "سَلام يا.", "سَلَاَم.",
    ],
    "شكراً": [
        "شُكْرَن.", "شُكْرًا.", "شُكرن.", "شُكْرَان.", "شُكْراً.",
    ],
    "عفواً": [
        "عَفْوَن.", "عَفْوًا.", "عَفوَن.", "عَفْواً.",
    ],
    "لو سمحت": [
        "لَو سَمَحْت.", "لَو سَمَحتْ.", "لَو سَمَحِت.", "لو سَمَحْت.",
    ],
    "إزيك": [
        "إِزَّيَّك.", "إِزيِك.", "إِزَّيك.", "إزيك.", "إِزَّيِّك.",
    ],
    "كويس": [
        "كُوَيِّس.", "كُوَيِّسْ.", "كْوَيِّس.", "كويس.",
    ],
    "مش كويس": [
        "مُش كُوَيِّس.", "مِش كُوَيِّس.", "مُش كْوَيِّس.", "مش كويس.",
    ],
    "عايز": [
        "عَايِز.", "عايِز.", "عَايْز.", "عايز.",
    ],
    "روح": [
        "رُوح.", "رُوحْ.", "رُوُح.", "روح.",
    ],
    "تعال": [
        "تَعَال.", "تَعَالْ.", "تَعَاَل.", "تعال.",
    ],
    "آكل": [
        "آكُل.", "آكِل.", "آكُلْ.", "أَكُل.",
    ],
    "أكل": [
        "أَكُل.", "أَكِل.", "أَكُلْ.", "آكُل.",
    ],
    "أشرب": [
        "أَشْرَب.", "أَشْرِب.", "أَشْرَبْ.", "اشرب.",
    ],
    "نام": [
        "نَام.", "نَامْ.", "نَاام.", "نام.",
    ],
    "معايا": [
        "مَعَايَا.", "مَعَايا.", "مَعَايَاْ.", "معايا.",
    ],
    "فين": [
        "فِين.", "فِينْ.", "فين.", "فِيِن.",
    ],
    "كلام": [
        "كَلَام.", "كِلام.", "كَلَامْ.", "كلام.",
    ],
    "فلوس": [
        "فُلُوس.", "فُلوس.", "فُلُوسْ.", "فلوس.",
    ],
    "بيت": [
        "بَيْت.", "بِيْت.", "بَيْتْ.", "بيت.",
    ],
    "عربية": [
        "عَرَبِيَّة.", "عَرَبِيّه.", "عَرَبِيَّهْ.", "عربية.",
    ],
    "بحبك": [
        "بَحِبَّك.", "بَحَبَّك.", "بَحِبك.", "بحبك.",
    ],
    "مبروك": [
        "مَبْرُوك.", "مَبْروك.", "مَبْرُوكْ.", "مبروك.",
    ],
    "يلا": [
        "يَلَّا.", "يَلا.", "يَلَّاْ.", "يلا.",
    ],
    "تمام": [
        "تَمَام.", "تَمَاام.", "تَمَامْ.", "تمام.",
    ],
    "آسف": [
        "آسِف.", "أَاسِف.", "آسِفْ.", "اسف.", "آسف.",
    ],
}


def with_period(text: str) -> str:
    t = text.strip()
    if t.endswith((".", "!", "?")):
        return t
    return t + "."


def word_file_slug(word: str) -> str:
    """Filename slug: spaces -> underscore; strip tatweel for filenames."""
    s = word.strip().replace(" ", "_")
    for ch in "ًٌٍَُِّْ":
        s = s.replace(ch, "")
    return s


def comparison_mp3_name(word: str, version: int) -> str:
    return f"word_{word_file_slug(word)}_v{version:02d}.mp3"


words_for_tts = [w for w in unique_words if w not in NUMBER_WORDS]
missing_variants = [w for w in words_for_tts if w not in WORD_VARIANTS]
if missing_variants:
    print("Words without explicit variants (using plain + period fallback):", missing_variants)
    for w in missing_variants:
        WORD_VARIANTS[w] = [with_period(w)]

print("Number words (reuse approved audio, skip TTS):", sorted(NUMBER_WORDS & set(unique_words)))
print("Word targets for TTS:", len(words_for_tts))
print("Total variants to synthesize:", sum(len(WORD_VARIANTS[w]) for w in words_for_tts))

Number words (reuse approved audio, skip TTS): ['أربعة', 'اتنين', 'تسعة', 'تلاتة', 'تمانية', 'خمسة', 'سبعة', 'ستة', 'عشرة', 'واحد']
Word targets for TTS: 25
Total variants to synthesize: 104


## Synthesis Helpers

Async Edge TTS MP3 writer and logging helpers.

Number audio is intentionally **not** copied into the words experiment folders because approved numbers already live in `generated_audio_words/edge_tts_numbers_mapping_v1/`.

In [6]:
GENERATION_LOG: list[dict] = []


async def synth_edge_mp3(text: str, mp3_path: Path, voice: str | None = None) -> float:
    mp3_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    last_exc = None
    for attempt in range(4):
        try:
            comm = edge_tts.Communicate(text, voice or EDGE_VOICE)
            await comm.save(str(mp3_path))
            if mp3_path.is_file() and mp3_path.stat().st_size > 0:
                return time.perf_counter() - t0
        except Exception as exc:
            last_exc = exc
            if mp3_path.is_file() and mp3_path.stat().st_size == 0:
                mp3_path.unlink(missing_ok=True)
            await asyncio.sleep(1.5 * (attempt + 1))
    if last_exc:
        raise last_exc
    raise RuntimeError("synthesis produced empty file")


async def generate_word_variant(word: str, variant: str, mp3_path: Path) -> dict | None:
    tts_text = with_period(variant)
    try:
        infer_s = await synth_edge_mp3(tts_text, mp3_path)
    except Exception as exc:
        print(f"original word: {word}")
        print(f"variant text sent to TTS: {tts_text}")
        print(f"output path: {mp3_path}")
        print(f"FAILED: {exc}")
        print("---")
        return None
    if not mp3_path.is_file() or mp3_path.stat().st_size == 0:
        print(f"original word: {word}")
        print(f"variant text sent to TTS: {tts_text}")
        print(f"output path: {mp3_path}")
        print("FAILED: empty or missing file")
        print("---")
        return None
    rel = mp3_path.relative_to(BASE_DIR)
    print(f"original word: {word}")
    print(f"variant text sent to TTS: {tts_text}")
    print(f"output path: {rel}")
    print("---")
    return {
        "word": word,
        "variant": variant,
        "tts_text": tts_text,
        "mp3_path": mp3_path,
        "infer_s": infer_s,
    }


def run_async(coro):
    return asyncio.get_event_loop().run_until_complete(coro)


def find_approved_number_mp3(word: str) -> Path | None:
    patterns = NUMBER_APPROVED_GLOB.get(word, [f"*{word_file_slug(word)}*"])
    for root in APPROVED_NUMBERS_DIRS:
        if not root.is_dir():
            continue
        for pat in patterns:
            matches = sorted(root.glob(pat))
            matches = [m for m in matches if m.suffix.lower() == ".mp3"]
            if matches:
                return matches[0]
    return None


async def copy_approved_numbers():
    for word in sorted(NUMBER_WORDS & set(unique_words)):
        src = find_approved_number_mp3(word)
        if src is None:
            print(f"[numbers] no approved file for {word!r} (skipped copy)")
            continue
        dest = DIR_CHOSEN / f"word_{word_file_slug(word)}_approved.mp3"
        shutil.copy2(src, dest)
        print(f"[numbers] copied {src.relative_to(BASE_DIR)} -> {dest.relative_to(BASE_DIR)}")


COPY_APPROVED_NUMBERS_INTO_WORDS = False

if COPY_APPROVED_NUMBERS_INTO_WORDS:
    run_async(copy_approved_numbers())
else:
    print("Skipping number audio copy: approved numbers stay in generated_audio_words/edge_tts_numbers_mapping_v1/.")

Skipping number audio copy: approved numbers stay in generated_audio_words/edge_tts_numbers_mapping_v1/.


## Generate Comparison MP3s

Synthesize every word variant into `generated_audio_words/edge_tts_words/comparison/` using the naming pattern `word_<slug>_vNN.mp3`.

In [7]:
async def generate_all_word_variants():
    for word in words_for_tts:
        variants = WORD_VARIANTS.get(word, [with_period(word)])
        for idx, variant in enumerate(variants, start=1):
            fname = comparison_mp3_name(word, idx)
            mp3_path = DIR_COMPARISON / fname
            entry = await generate_word_variant(word, variant, mp3_path)
            if entry:
                entry["filename"] = fname
                GENERATION_LOG.append(entry)


REGENERATE_ROUND1 = False  # set True only to rebuild comparison/ from scratch

if REGENERATE_ROUND1:
    print(f"Generating {sum(len(WORD_VARIANTS[w]) for w in words_for_tts)} comparison files ...")
    run_async(generate_all_word_variants())
    print(f"Done. Generated: {len(GENERATION_LOG)} files in {DIR_COMPARISON.relative_to(BASE_DIR)}")
else:
    print("Skipping round-1 regeneration (REGENERATE_ROUND1=False).")
    print(f"Existing comparison files: {len(list(DIR_COMPARISON.glob('*.mp3')))}")

Skipping round-1 regeneration (REGENERATE_ROUND1=False).
Existing comparison files: 87


## Optional Cut Experiments

If a variant has trailing unwanted silence/noise, trim the tail with `pydub` (only when `pydub` is available). Outputs go to `generated_audio_words/edge_tts_words/cut/`.

In [8]:
# Words flagged for optional tail trim (extend if listening shows extra tail sound)
CUT_CANDIDATES: dict[str, list[tuple[str, float]]] = {
    # word_slug_prefix: [(source_comparison_filename, end_trim_seconds), ...]
}

try:
    from pydub import AudioSegment
    PYDUB_OK = True
except ImportError:
    PYDUB_OK = False
    print("pydub not installed — skipping cut experiments (comparison files unchanged).")

if PYDUB_OK and CUT_CANDIDATES:
    for word, cuts in CUT_CANDIDATES.items():
        for src_name, trim_sec in cuts:
            src = DIR_COMPARISON / src_name
            if not src.is_file():
                print(f"skip cut, missing: {src}")
                continue
            audio = AudioSegment.from_mp3(src)
            trimmed = audio[: max(0, len(audio) - int(trim_sec * 1000))]
            out = DIR_CUT / src_name.replace(".mp3", f"_cut_{trim_sec:.2f}.mp3")
            trimmed.export(out, format="mp3")
            print(f"cut: {src.name} -> {out.relative_to(BASE_DIR)}")
else:
    print("No cut candidates configured (listen first, then add to CUT_CANDIDATES).")

No cut candidates configured (listen first, then add to CUT_CANDIDATES).


## Playback Widgets

Display IPython audio players grouped by original word for quick listening comparisons.

In [9]:
from collections import defaultdict

by_word: dict[str, list[dict]] = defaultdict(list)
for entry in GENERATION_LOG:
    by_word[entry["word"]].append(entry)

for word in words_for_tts:
    entries = sorted(by_word.get(word, []), key=lambda e: e["filename"])
    if not entries:
        continue
    print("=" * 60)
    print(f"WORD: {word}  ({len(entries)} variants)")
    print("=" * 60)
    for e in entries:
        print(f"  {e['filename']}  |  TTS: {e['tts_text']}")
        display(IPA(filename=str(e["mp3_path"]), autoplay=False))
    print()

## Round 2 — comparison2 (pronunciation improvements)

Keep approved files in `chosen/` and round-1 files in `comparison/` unchanged.

Generate **new** variants only for words that still need work, into:

`generated_audio_words/edge_tts_words/comparison2/`

Approved stems (do not regenerate):

`word_إزيك_v01`, `word_آسف_v01`, `word_أشرب_v03`, `word_آكل_v01`, `word_بحبك_v03`, `word_بيت_v01`, `word_سلام_v01`, `word_عايز_v01`, `word_عربية_v03`, `word_عفوا_v01`, `word_فلوس_v04`, `word_فين_v04`, `word_كلام_v03`, `word_مبروك_v03`, `word_مش_كويس_v01`, `word_نام_v01`

In [10]:
# Round-2 words and Egyptian-friendly variant spellings (comparison2 only)
CHOSEN_STEMS = {
    "word_إزيك_v01", "word_آسف_v01", "word_أشرب_v03", "word_آكل_v01",
    "word_بحبك_v03", "word_بيت_v01", "word_سلام_v01", "word_عايز_v01",
    "word_عربية_v03", "word_عفوا_v01", "word_فلوس_v04", "word_فين_v04",
    "word_كلام_v03", "word_مبروك_v03", "word_مش_كويس_v01", "word_نام_v01",
}

WORD_VARIANTS_ROUND2: dict[str, list[str]] = {
    "تعال": [
        "تَعَالَى.", "تَعَالَا.", "تعالى.", "تعالا.", "ta3ala.", "taala.",
    ],
    "تمام": [
        "تَمَاام.", "تَمَام.", "tamaam.", "tamām.", "تَمَاَم.",
    ],
    "شكراً": [
        "شُكْرَان.", "شُكران.", "shokran.", "shokrān.", "شُكْرَن.",
    ],
    "كويس": [
        "كُوَيِّس.", "كُوَايِس.", "kowayes.", "kowayyis.", "كوَايس.",
    ],
    "معايا": [
        "مَعَايَا.", "مَعَايا.", "ma3aya.", "maaya.", "مَعَيَّا.", "مَعايا.",
    ],
    "يلا": [
        "يَلَّا.", "يَلَا.", "yalla.", "yallaa.", "يَللَا.",
    ],
}

ROUND2_WORDS = list(WORD_VARIANTS_ROUND2.keys())
GENERATION_LOG2: list[dict] = []

print("Round-2 words:", ROUND2_WORDS)
print("Variants to synthesize:", sum(len(v) for v in WORD_VARIANTS_ROUND2.values()))
print("Chosen stems locked:", len(CHOSEN_STEMS))

Round-2 words: ['تعال', 'تمام', 'شكراً', 'كويس', 'معايا', 'يلا']
Variants to synthesize: 32
Chosen stems locked: 16


## Generate comparison2 MP3s

Synthesize round-2 variants into `comparison2/` (MP3 only). Does not touch `chosen/` or round-1 `comparison/`.

In [11]:
async def generate_round2_variants():
    GENERATION_LOG2.clear()
    for word in ROUND2_WORDS:
        variants = WORD_VARIANTS_ROUND2[word]
        for idx, variant in enumerate(variants, start=1):
            fname = comparison_mp3_name(word, idx)
            stem = fname.removesuffix(".mp3")
            if stem in CHOSEN_STEMS:
                print(f"SKIP (chosen): {stem}")
                continue
            mp3_path = DIR_COMPARISON2 / fname
            entry = await generate_word_variant(word, variant, mp3_path)
            if entry:
                entry["filename"] = fname
                GENERATION_LOG2.append(entry)


REGENERATE_ROUND2 = False

if REGENERATE_ROUND2:
    print(f"Generating round-2 files into {DIR_COMPARISON2.relative_to(BASE_DIR)} ...")
    run_async(generate_round2_variants())
    print(f"Done. Generated: {len(GENERATION_LOG2)} files")
else:
    print("Skipping round-2 regeneration (REGENERATE_ROUND2=False).")
    print(f"Existing comparison2 files: {len(list(DIR_COMPARISON2.glob('*.mp3')))}")

Skipping round-2 regeneration (REGENERATE_ROUND2=False).
Existing comparison2 files: 31


## comparison2 cuts (روح, لو سمحت)

Trim ~0.6s from the tail of round-1 comparison files. Save to `comparison2/cut/` without overwriting originals.

In [12]:
COMPARISON2_CUT_JOBS = [
    ("word_روح_v01.mp3", "word_روح_v01_cut_0_6.mp3", 0.6),
    ("word_لو_سمحت_v01.mp3", "word_لو_سمحت_v01_cut_0_6.mp3", 0.6),
]

_ensure("pydub")
from pydub import AudioSegment

CUT_LOG2: list[dict] = []

for src_name, out_name, trim_sec in COMPARISON2_CUT_JOBS:
    src = DIR_COMPARISON / src_name
    if not src.is_file():
        print(f"SKIP cut — missing source: {src.relative_to(BASE_DIR)}")
        continue
    audio = AudioSegment.from_mp3(src)
    orig_ms = len(audio)
    orig_sec = orig_ms / 1000.0
    trim_ms = int(trim_sec * 1000)
    trimmed = audio[: max(0, orig_ms - trim_ms)]
    cut_sec = len(trimmed) / 1000.0
    out = DIR_COMPARISON2_CUT / out_name
    trimmed.export(out, format="mp3")
    rel = out.relative_to(BASE_DIR)
    print(f"source: {src.relative_to(BASE_DIR)}")
    print(f"original duration: {orig_sec:.3f}s")
    print(f"cut duration: {cut_sec:.3f}s (trimmed ~{trim_sec}s from tail)")
    print(f"saved path: {rel}")
    print("---")
    CUT_LOG2.append({"src": src, "out": out, "orig_sec": orig_sec, "cut_sec": cut_sec})

print(f"Cut files written: {len(CUT_LOG2)}")

source: generated_audio_words\edge_tts_words\comparison\word_روح_v01.mp3

original duration: 1.680s
cut duration: 1.080s (trimmed ~0.6s from tail)
saved path: generated_audio_words\edge_tts_words\comparison2\cut\word_روح_v01_cut_0_6.mp3
---


source: generated_audio_words\edge_tts_words\comparison\word_لو_سمحت_v01.mp3
original duration: 1.944s
cut duration: 1.344s (trimmed ~0.6s from tail)
saved path: generated_audio_words\edge_tts_words\comparison2\cut\word_لو_سمحت_v01_cut_0_6.mp3
---
Cut files written: 2


## comparison2 playback

Listen to round-2 TTS variants and cut versions grouped by word.

In [13]:
from collections import defaultdict

by_word2: dict[str, list[dict]] = defaultdict(list)
for entry in GENERATION_LOG2:
    by_word2[entry["word"]].append(entry)

for word in ROUND2_WORDS:
    entries = sorted(by_word2.get(word, []), key=lambda e: e["filename"])
    if not entries:
        continue
    print("=" * 60)
    print(f"ROUND 2 — WORD: {word}  ({len(entries)} variants)")
    print("=" * 60)
    for e in entries:
        print(f"  {e['filename']}  |  TTS: {e['tts_text']}")
        display(IPA(filename=str(e["mp3_path"]), autoplay=False))
    print()

if CUT_LOG2:
    print("=" * 60)
    print("ROUND 2 — CUTS")
    print("=" * 60)
    for c in CUT_LOG2:
        print(f"  {c['out'].name}  ({c['cut_sec']:.3f}s)")
        display(IPA(filename=str(c["out"]), autoplay=False))

ROUND 2 — CUTS
  word_روح_v01_cut_0_6.mp3  (1.080s)


  word_لو_سمحت_v01_cut_0_6.mp3  (1.344s)


## Round 3 — comparison3 (0.6s cuts, chosen move, يلا)

- Cut selected clips to **~0.6 seconds** (keep first 0.6s) → `comparison3/cut/`
- Move approved `word_تعال_v01` → `chosen/`
- New **يلا** variants (مشدود لام) → `comparison3/`

In [14]:
DIR_COMPARISON3.mkdir(parents=True, exist_ok=True)
DIR_COMPARISON3_CUT.mkdir(parents=True, exist_ok=True)
DIR_CHOSEN.mkdir(parents=True, exist_ok=True)

TARGET_CLIP_SEC = 0.6

COMPARISON3_CUT_STEMS = [
    "word_روح_v01",
    "word_لو_سمحت_v01",
    "word_تمام_v02",
    "word_شكرا_v05",
    "word_كويس_v01",
    "word_معايا_v01",
]

YALLA_VARIANTS_ROUND3 = [
    "يَلَّا.", "يَللَا.", "يالا.", "yalla.", "yallaa.", "yal-la.", "ya lla.",
]

COMPARISON_SEARCH_DIRS = (DIR_COMPARISON2, DIR_COMPARISON, DIR_COMPARISON3)


def find_source_mp3(stem: str) -> Path | None:
    name = stem if stem.endswith(".mp3") else f"{stem}.mp3"
    for folder in COMPARISON_SEARCH_DIRS:
        path = folder / name
        if path.is_file() and path.stat().st_size > 0:
            return path
    return None


_ensure("pydub")
from pydub import AudioSegment

CUT_LOG3: list[dict] = []
MOVED_TO_CHOSEN: list[str] = []
GENERATION_LOG3: list[dict] = []

print("=" * 60)
print("CUT TO ~0.6s → comparison3/cut/")
print("=" * 60)

for stem in COMPARISON3_CUT_STEMS:
    src = find_source_mp3(stem)
    if src is None:
        print(f"MISSING source for {stem}")
        continue
    audio = AudioSegment.from_mp3(src)
    orig_sec = len(audio) / 1000.0
    cut_audio = audio[: int(TARGET_CLIP_SEC * 1000)]
    cut_sec = len(cut_audio) / 1000.0
    out_name = f"{stem}_cut_0_6.mp3"
    out = DIR_COMPARISON3_CUT / out_name
    cut_audio.export(out, format="mp3")
    rel = out.relative_to(BASE_DIR)
    print(f"source: {src.relative_to(BASE_DIR)}")
    print(f"original duration: {orig_sec:.3f}s")
    print(f"cut duration: {cut_sec:.3f}s")
    print(f"saved path: {rel}")
    print("---")
    CUT_LOG3.append({"stem": stem, "src": src, "out": out, "orig_sec": orig_sec, "cut_sec": cut_sec})

print(f"Files cut: {len(CUT_LOG3)}")

print("=" * 60)
print("MOVE TO chosen/")
print("=" * 60)

TAAL_CHOSEN_STEM = "word_تعال_v01"
taal_dest = DIR_CHOSEN / f"{TAAL_CHOSEN_STEM}.mp3"
taal_src = find_source_mp3(TAAL_CHOSEN_STEM)
if taal_src is None:
    print(f"MISSING: {TAAL_CHOSEN_STEM}")
else:
    if taal_dest.is_file():
        taal_dest.unlink()
    shutil.move(str(taal_src), str(taal_dest))
    MOVED_TO_CHOSEN.append(str(taal_dest.relative_to(BASE_DIR)))
    print(f"moved to chosen: {taal_dest.relative_to(BASE_DIR)}")
    for folder in (DIR_COMPARISON, DIR_COMPARISON2):
        leftover = folder / f"{TAAL_CHOSEN_STEM}.mp3"
        if leftover.is_file():
            leftover.unlink()
            print(f"removed from {folder.relative_to(BASE_DIR)}: {TAAL_CHOSEN_STEM}.mp3")

print(f"Files moved to chosen: {MOVED_TO_CHOSEN}")

print("=" * 60)
print("يلا variants → comparison3/")
print("=" * 60)


async def generate_yalla_round3():
    GENERATION_LOG3.clear()
    word = "يلا"
    for idx, variant in enumerate(YALLA_VARIANTS_ROUND3, start=1):
        fname = comparison_mp3_name(word, idx)
        mp3_path = DIR_COMPARISON3 / fname
        entry = await generate_word_variant(word, variant, mp3_path)
        if entry:
            entry["filename"] = fname
            GENERATION_LOG3.append(entry)


run_async(generate_yalla_round3())
print(f"يلا variants generated: {len(GENERATION_LOG3)}")
for e in GENERATION_LOG3:
    print(f"  {e['filename']} -> {e['mp3_path'].relative_to(BASE_DIR)}")

CUT TO ~0.6s → comparison3/cut/


source: generated_audio_words\edge_tts_words\comparison\word_روح_v01.mp3
original duration: 1.680s
cut duration: 0.600s
saved path: generated_audio_words\edge_tts_words\comparison3\cut\word_روح_v01_cut_0_6.mp3
---


source: generated_audio_words\edge_tts_words\comparison\word_لو_سمحت_v01.mp3
original duration: 1.944s
cut duration: 0.600s
saved path: generated_audio_words\edge_tts_words\comparison3\cut\word_لو_سمحت_v01_cut_0_6.mp3
---


source: generated_audio_words\edge_tts_words\comparison2\word_تمام_v02.mp3
original duration: 1.632s
cut duration: 0.600s
saved path: generated_audio_words\edge_tts_words\comparison3\cut\word_تمام_v02_cut_0_6.mp3
---


source: generated_audio_words\edge_tts_words\comparison2\word_شكرا_v05.mp3
original duration: 1.704s
cut duration: 0.600s
saved path: generated_audio_words\edge_tts_words\comparison3\cut\word_شكرا_v05_cut_0_6.mp3
---


...

original word: يلا
variant text sent to TTS: يالا.
output path: generated_audio_words\edge_tts_words\comparison3\word_يلا_v03.mp3
---


original word: يلا
variant text sent to TTS: yalla.
output path: generated_audio_words\edge_tts_words\comparison3\word_يلا_v04.mp3
---


original word: يلا
variant text sent to TTS: yallaa.
output path: generated_audio_words\edge_tts_words\comparison3\word_يلا_v05.mp3
---


original word: يلا
variant text sent to TTS: yal-la.
output path: generated_audio_words\edge_tts_words\comparison3\word_يلا_v06.mp3
---


original word: يلا
variant text sent to TTS: ya lla.
output path: generated_audio_words\edge_tts_words\comparison3\word_يلا_v07.mp3
---
يلا variants generated: 7
  word_يلا_v01.mp3 -> generated_audio_words\edge_tts_words\comparison3\word_يلا_v01.mp3
  word_يلا_v02.mp3 -> generated_audio_words\edge_tts_words\comparison3\word_يلا_v02.mp3
  word_يلا_v03.mp3 -> generated_audio_words\edge_tts_words\comparison3\word_يلا_v03.mp3
  word_يلا_v04.mp3 -> generated_audio_words\edge_tts_words\comparison3\word_يلا_v04.mp3
  word_يلا_v05.mp3 -> generated_audio_words\edge_tts_words\comparison3\word_يلا_v05.mp3
  word_يلا_v06.mp3 -> generated_audio_words\edge_tts_words\comparison3\word_يلا_v06.mp3
  word_يلا_v07.mp3 -> generated_audio_words\edge_tts_words\comparison3\word_يلا_v07.mp3


## comparison3 playback

Listen to 0.6s cuts and new يلا variants.

In [15]:
if CUT_LOG3:
    print("=" * 60)
    print("CUT CLIPS (~0.6s)")
    print("=" * 60)
    for c in CUT_LOG3:
        print(f"  {c['out'].name}  (was {c['orig_sec']:.3f}s → {c['cut_sec']:.3f}s)")
        display(IPA(filename=str(c["out"]), autoplay=False))

if GENERATION_LOG3:
    print("=" * 60)
    print("يلا — comparison3")
    print("=" * 60)
    for e in GENERATION_LOG3:
        print(f"  {e['filename']}  |  TTS: {e['tts_text']}")
        display(IPA(filename=str(e["mp3_path"]), autoplay=False))

taal_chosen = DIR_CHOSEN / "word_تعال_v01.mp3"
if taal_chosen.is_file():
    print("=" * 60)
    print("CHOSEN: word_تعال_v01")
    print("=" * 60)
    display(IPA(filename=str(taal_chosen), autoplay=False))
else:
    print("word_تعال_v01 not found in chosen/")

CUT CLIPS (~0.6s)
  word_روح_v01_cut_0_6.mp3  (was 1.680s → 0.600s)


  word_لو_سمحت_v01_cut_0_6.mp3  (was 1.944s → 0.600s)


  word_تمام_v02_cut_0_6.mp3  (was 1.632s → 0.600s)


  word_شكرا_v05_cut_0_6.mp3  (was 1.704s → 0.600s)


  word_كويس_v01_cut_0_6.mp3  (was 1.656s → 0.600s)


...

  word_يلا_v04.mp3  |  TTS: yalla.


  word_يلا_v05.mp3  |  TTS: yallaa.


  word_يلا_v06.mp3  |  TTS: yal-la.


  word_يلا_v07.mp3  |  TTS: ya lla.


CHOSEN: word_تعال_v01


## Move Approved Word Files to chosen/

Use the corrected base folder:

`generated_audio_words/edge_tts_words/chosen/`

Move the approved word MP3s into `chosen/` and remove duplicate copies from `comparison/`, `comparison2/`, and `comparison3/` so each approved stem exists only in `chosen/`.

In [16]:
APPROVED_WORD_STEMS = [
    "word_إزيك_v01",
    "word_آسف_v01",
    "word_أشرب_v03",
    "word_آكل_v01",
    "word_بحبك_v03",
    "word_بيت_v01",
    "word_سلام_v01",
    "word_عايز_v01",
    "word_عربية_v03",
    "word_عفوا_v01",
    "word_فلوس_v04",
    "word_فين_v04",
    "word_كلام_v03",
    "word_مبروك_v03",
    "word_مش_كويس_v01",
    "word_نام_v01",
    "word_تعال_v01",
]

CHOSEN_SOURCE_DIRS = [DIR_COMPARISON, DIR_COMPARISON2, DIR_COMPARISON3]
APPROVED_FALLBACK_TTS = {
    # Source copy may already have been removed after the previous approval step.
    "word_تعال_v01": "تَعَالَى.",
}
DIR_CHOSEN.mkdir(parents=True, exist_ok=True)

moved_to_chosen: list[dict] = []
already_in_chosen: list[str] = []
removed_duplicates: list[dict] = []
missing_approved: list[str] = []

print("chosen folder:", DIR_CHOSEN.relative_to(BASE_DIR))
print("source folders:")
for folder in CHOSEN_SOURCE_DIRS:
    print(" -", folder.relative_to(BASE_DIR))
print("=" * 60)

for stem in APPROVED_WORD_STEMS:
    filename = f"{stem}.mp3"
    dest = DIR_CHOSEN / filename
    source = None

    if dest.is_file() and dest.stat().st_size > 0:
        already_in_chosen.append(filename)
    else:
        for folder in CHOSEN_SOURCE_DIRS:
            candidate = folder / filename
            if candidate.is_file() and candidate.stat().st_size > 0:
                source = candidate
                break

        if source is None:
            fallback_tts = APPROVED_FALLBACK_TTS.get(stem)
            if fallback_tts:
                run_async(synth_edge_mp3(fallback_tts, dest))
                moved_to_chosen.append({
                    "file": filename,
                    "source": "regenerated approved fallback",
                    "destination": DIR_CHOSEN.relative_to(BASE_DIR),
                })
                print(f"RESTORED: {filename}")
                print(f"  source folder: regenerated approved fallback")
                print(f"  destination folder: {DIR_CHOSEN.relative_to(BASE_DIR)}")
            else:
                missing_approved.append(filename)
                print(f"MISSING: {filename}")
                continue
        else:
            if dest.is_file():
                dest.unlink()
            shutil.move(str(source), str(dest))
            moved_to_chosen.append({
                "file": filename,
                "source": source.parent.relative_to(BASE_DIR),
                "destination": DIR_CHOSEN.relative_to(BASE_DIR),
            })
            print(f"MOVED: {filename}")
            print(f"  source folder: {source.parent.relative_to(BASE_DIR)}")
            print(f"  destination folder: {DIR_CHOSEN.relative_to(BASE_DIR)}")

    if dest.is_file() and dest.stat().st_size > 0:
        for folder in CHOSEN_SOURCE_DIRS:
            duplicate = folder / filename
            if duplicate.is_file():
                duplicate.unlink()
                removed_duplicates.append({
                    "file": filename,
                    "removed_from": folder.relative_to(BASE_DIR),
                })
                print(f"REMOVED duplicate: {folder.relative_to(BASE_DIR) / filename}")

print("=" * 60)
print("files moved:", len(moved_to_chosen))
for item in moved_to_chosen:
    print(f" - {item['file']} | {item['source']} -> {item['destination']}")
print("already in chosen:", len(already_in_chosen))
for filename in already_in_chosen:
    print(f" - {filename}")
print("duplicates removed:", len(removed_duplicates))
for item in removed_duplicates:
    print(f" - {item['file']} from {item['removed_from']}")
print("missing files:", len(missing_approved))
for filename in missing_approved:
    print(f" - {filename}")

all_chosen_exist = all((DIR_CHOSEN / f"{stem}.mp3").is_file() for stem in APPROVED_WORD_STEMS)
print("chosen folder exists:", DIR_CHOSEN.is_dir())
print("all listed files exist in chosen:", all_chosen_exist)

chosen folder: generated_audio_words\edge_tts_words\chosen
source folders:
 - generated_audio_words\edge_tts_words\comparison
 - generated_audio_words\edge_tts_words\comparison2
 - generated_audio_words\edge_tts_words\comparison3
files moved: 0
already in chosen: 17
 - word_إزيك_v01.mp3
 - word_آسف_v01.mp3
 - word_أشرب_v03.mp3
 - word_آكل_v01.mp3
 - word_بحبك_v03.mp3
 - word_بيت_v01.mp3
 - word_سلام_v01.mp3
 - word_عايز_v01.mp3
 - word_عربية_v03.mp3
 - word_عفوا_v01.mp3
 - word_فلوس_v04.mp3
 - word_فين_v04.mp3
 - word_كلام_v03.mp3
 - word_مبروك_v03.mp3
 - word_مش_كويس_v01.mp3
 - word_نام_v01.mp3
 - word_تعال_v01.mp3
duplicates removed: 0
missing files: 0
chosen folder exists: True
all listed files exist in chosen: True


## Chosen Playback Examples

Display a few chosen examples to confirm the approved files are playable from `generated_audio_words/edge_tts_words/chosen/`.

In [17]:
CHOSEN_PLAYBACK_EXAMPLES = [
    "word_إزيك_v01.mp3",
    "word_سلام_v01.mp3",
    "word_تعال_v01.mp3",
    "word_مش_كويس_v01.mp3",
    "word_نام_v01.mp3",
]

print("Chosen playback examples:")
for filename in CHOSEN_PLAYBACK_EXAMPLES:
    path = DIR_CHOSEN / filename
    print(f" - {path.relative_to(BASE_DIR)} | exists: {path.is_file()}")
    if path.is_file():
        display(IPA(filename=str(path), autoplay=False))

Chosen playback examples:
 - generated_audio_words\edge_tts_words\chosen\word_إزيك_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_سلام_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_تعال_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_مش_كويس_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_نام_v01.mp3 | exists: True


## Cleanup Number Audio From Word Experiment Folders

Remove number-related MP3s from `generated_audio_words/edge_tts_words/` and its comparison/chosen folders.

This keeps only actual word pronunciation experiment files under the words audio tree; approved numbers remain in `generated_audio_words/edge_tts_numbers_mapping_v1/`.

In [2]:
NUMBER_AUDIO_WORD_STEMS = {
    "word_واحد_approved",
    "word_اتنين_approved",
    "word_تلاتة_approved",
    "word_أربعة_approved",
    "word_خمسة_approved",
    "word_ستة_approved",
    "word_سبعة_approved",
    "word_تمانية_approved",
    "word_تسعة_approved",
    "word_عشرة_approved",
}

NUMBER_AUDIO_PREFIXES = ("num_", "number_")
WORD_AUDIO_FOLDERS = [
    DIR_WORDS,
    DIR_COMPARISON,
    DIR_COMPARISON2,
    DIR_COMPARISON2_CUT,
    DIR_COMPARISON3,
    DIR_COMPARISON3_CUT,
    DIR_CHOSEN,
    DIR_CUT,
]

removed_number_files: list[Path] = []
kept_word_files: list[Path] = []

for folder in WORD_AUDIO_FOLDERS:
    if not folder.exists():
        continue
    for mp3_path in sorted(folder.glob("*.mp3")):
        stem = mp3_path.stem
        is_number_audio = stem.startswith(NUMBER_AUDIO_PREFIXES) or stem in NUMBER_AUDIO_WORD_STEMS
        if is_number_audio:
            mp3_path.unlink()
            removed_number_files.append(mp3_path)
        else:
            kept_word_files.append(mp3_path)

# Include word files in nested folders under edge_tts_words that were not listed above.
for mp3_path in sorted(DIR_WORDS.rglob("*.mp3")):
    if mp3_path not in kept_word_files and mp3_path not in removed_number_files:
        stem = mp3_path.stem
        is_number_audio = stem.startswith(NUMBER_AUDIO_PREFIXES) or stem in NUMBER_AUDIO_WORD_STEMS
        if is_number_audio:
            mp3_path.unlink()
            removed_number_files.append(mp3_path)
        else:
            kept_word_files.append(mp3_path)

remaining_number_files = []
remaining_word_files = []
for mp3_path in sorted(DIR_WORDS.rglob("*.mp3")):
    stem = mp3_path.stem
    if stem.startswith(NUMBER_AUDIO_PREFIXES) or stem in NUMBER_AUDIO_WORD_STEMS:
        remaining_number_files.append(mp3_path)
    else:
        remaining_word_files.append(mp3_path)

print("removed number files:")
if removed_number_files:
    for path in removed_number_files:
        print(" -", path.relative_to(BASE_DIR))
else:
    print(" - none")

print("\nkept word files:")
for path in remaining_word_files:
    print(" -", path.relative_to(BASE_DIR))

print("\nfinal remaining word file count:", len(remaining_word_files))
print("remaining number files:", len(remaining_number_files))
if remaining_number_files:
    for path in remaining_number_files:
        print(" -", path.relative_to(BASE_DIR))

print("word files still exist:", len(remaining_word_files) > 0)

removed number files:
 - none

kept word files:
 - generated_audio_words\edge_tts_words\chosen\word_آسف_v01.mp3
 - generated_audio_words\edge_tts_words\chosen\word_آكل_v01.mp3
 - generated_audio_words\edge_tts_words\chosen\word_أشرب_v03.mp3
 - generated_audio_words\edge_tts_words\chosen\word_إزيك_v01.mp3
 - generated_audio_words\edge_tts_words\chosen\word_بحبك_v03.mp3
 - generated_audio_words\edge_tts_words\chosen\word_بيت_v01.mp3
 - generated_audio_words\edge_tts_words\chosen\word_تعال_v01.mp3
 - generated_audio_words\edge_tts_words\chosen\word_سلام_v01.mp3
 - generated_audio_words\edge_tts_words\chosen\word_عايز_v01.mp3
 - generated_audio_words\edge_tts_words\chosen\word_عربية_v03.mp3
 - generated_audio_words\edge_tts_words\chosen\word_عفوا_v01.mp3
 - generated_audio_words\edge_tts_words\chosen\word_فلوس_v04.mp3
 - generated_audio_words\edge_tts_words\chosen\word_فين_v04.mp3
 - generated_audio_words\edge_tts_words\chosen\word_كلام_v03.mp3
 - generated_audio_words\edge_tts_words\chose

## Word Audio After Number Cleanup

Display a few remaining word files to confirm word audio is still available after removing number-related MP3s.

In [3]:
CLEANUP_PLAYBACK_EXAMPLES = [
    "chosen/word_سلام_v01.mp3",
    "chosen/word_تعال_v01.mp3",
    "chosen/word_إزيك_v01.mp3",
    "chosen/word_مش_كويس_v01.mp3",
    "chosen/word_نام_v01.mp3",
]

print("Word playback examples after cleanup:")
for rel in CLEANUP_PLAYBACK_EXAMPLES:
    path = DIR_WORDS / rel
    print(f" - {path.relative_to(BASE_DIR)} | exists: {path.is_file()}")
    if path.is_file():
        display(IPA(filename=str(path), autoplay=False))

Word playback examples after cleanup:
 - generated_audio_words\edge_tts_words\chosen\word_سلام_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_تعال_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_إزيك_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_مش_كويس_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_نام_v01.mp3 | exists: True


## Round 4 — comparison4 (recuts and يلا)

Use `generated_audio_words/` as the base folder.

- Move approved 0.6s cuts for `روح` and `تمام` into `chosen/`
- Create new 0.65s and 0.8s recuts in `comparison4/cut/`
- Generate new `يلا` variants in `comparison4/`

In [2]:
DIR_COMPARISON4.mkdir(parents=True, exist_ok=True)
DIR_COMPARISON4_CUT.mkdir(parents=True, exist_ok=True)
DIR_CHOSEN.mkdir(parents=True, exist_ok=True)

_ensure("pydub")
from pydub import AudioSegment

COMPARISON4_MOVE_TO_CHOSEN = [
    "word_روح_v01_cut_0_6",
    "word_تمام_v02_cut_0_6",
]

COMPARISON4_RECUT_JOBS = [
    ("word_معايا_v01", 0.65, "word_معايا_v01_cut_0_65.mp3"),
    ("word_كويس_v01", 0.65, "word_كويس_v01_cut_0_65.mp3"),
    ("word_شكرا_v05", 0.65, "word_شكرا_v05_cut_0_65.mp3"),
    ("word_لو_سمحت_v01", 0.80, "word_لو_سمحت_v01_cut_0_8.mp3"),
]

YALLA_VARIANTS_ROUND4 = [
    "يَلَّا.",
    "يَللَا.",
    "يَالَّا.",
    "يَلاّ.",
    "يالله.",
    "yalla.",
    "yallaa.",
    "yal-la.",
    "ya lla.",
    "yalla!",
]

CUT_SOURCE_DIRS = [DIR_COMPARISON3_CUT, DIR_COMPARISON2_CUT, DIR_CUT, DIR_COMPARISON4_CUT]
ORIGINAL_SOURCE_DIRS = [DIR_COMPARISON2, DIR_COMPARISON, DIR_COMPARISON3, DIR_CHOSEN]


def _mp3_name(stem: str) -> str:
    return stem if stem.endswith(".mp3") else f"{stem}.mp3"


def find_cut_source(stem: str) -> Path | None:
    name = _mp3_name(stem)
    for folder in CUT_SOURCE_DIRS:
        candidate = folder / name
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    chosen_candidate = DIR_CHOSEN / name
    if chosen_candidate.is_file() and chosen_candidate.stat().st_size > 0:
        return chosen_candidate
    return None


def find_original_source(stem: str) -> Path | None:
    name = _mp3_name(stem)
    for folder in ORIGINAL_SOURCE_DIRS:
        candidate = folder / name
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    return None


def ensure_edge_voice_for_round4() -> str:
    global EDGE_VOICE
    if "EDGE_VOICE" in globals() and EDGE_VOICE:
        return EDGE_VOICE

    async def _select_voice() -> str:
        voices = await edge_tts.list_voices()
        by_short = {v["ShortName"]: v for v in voices}
        for pref in ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]:
            if pref in by_short:
                return pref
        raise RuntimeError("No Egyptian Arabic Edge TTS voice found")

    EDGE_VOICE = asyncio.get_event_loop().run_until_complete(_select_voice())
    return EDGE_VOICE


async def synth_round4_mp3(text: str, mp3_path: Path, voice: str) -> float:
    mp3_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    last_exc = None
    for attempt in range(4):
        try:
            comm = edge_tts.Communicate(text, voice)
            await comm.save(str(mp3_path))
            if mp3_path.is_file() and mp3_path.stat().st_size > 0:
                return time.perf_counter() - t0
        except Exception as exc:
            last_exc = exc
            if mp3_path.is_file() and mp3_path.stat().st_size == 0:
                mp3_path.unlink(missing_ok=True)
            await asyncio.sleep(1.5 * (attempt + 1))
    if last_exc:
        raise last_exc
    raise RuntimeError("synthesis produced empty file")


COMPARISON4_MOVED_TO_CHOSEN: list[dict] = []
COMPARISON4_RECUT_LOG: list[dict] = []
COMPARISON4_YALLA_LOG: list[dict] = []
COMPARISON4_MISSING: list[str] = []

print("=" * 60)
print("MOVE APPROVED CUTS TO chosen/")
print("=" * 60)

for stem in COMPARISON4_MOVE_TO_CHOSEN:
    filename = _mp3_name(stem)
    dest = DIR_CHOSEN / filename
    src = find_cut_source(stem)

    if src is None:
        if dest.is_file():
            print(f"already in chosen: {dest.relative_to(BASE_DIR)}")
            COMPARISON4_MOVED_TO_CHOSEN.append({"file": filename, "source": "already in chosen", "destination": dest})
        else:
            print(f"MISSING move source: {filename}")
            COMPARISON4_MISSING.append(filename)
        continue

    if src.resolve() != dest.resolve():
        if dest.is_file():
            dest.unlink()
        shutil.move(str(src), str(dest))
        print(f"moved to chosen: {filename}")
        print(f"source folder: {src.parent.relative_to(BASE_DIR)}")
        print(f"destination folder: {DIR_CHOSEN.relative_to(BASE_DIR)}")
    else:
        print(f"already in chosen: {dest.relative_to(BASE_DIR)}")

    for folder in CUT_SOURCE_DIRS:
        duplicate = folder / filename
        if duplicate.is_file() and duplicate.resolve() != dest.resolve():
            duplicate.unlink()
            print(f"removed duplicate: {duplicate.relative_to(BASE_DIR)}")

    COMPARISON4_MOVED_TO_CHOSEN.append({"file": filename, "source": src.parent, "destination": dest})

print("=" * 60)
print("RECUT FILES INTO comparison4/cut/")
print("=" * 60)

for source_stem, target_sec, out_name in COMPARISON4_RECUT_JOBS:
    src = find_original_source(source_stem)
    if src is None:
        print(f"MISSING recut source: {source_stem}.mp3")
        COMPARISON4_MISSING.append(f"{source_stem}.mp3")
        continue

    audio = AudioSegment.from_mp3(src)
    original_sec = len(audio) / 1000.0
    cut_audio = audio[: int(target_sec * 1000)]
    cut_sec = len(cut_audio) / 1000.0
    out = DIR_COMPARISON4_CUT / out_name
    cut_audio.export(out, format="mp3")

    print(f"source: {src.relative_to(BASE_DIR)}")
    print(f"original duration: {original_sec:.3f}s")
    print(f"new cut duration: {cut_sec:.3f}s")
    print(f"saved path: {out.relative_to(BASE_DIR)}")
    print("---")
    COMPARISON4_RECUT_LOG.append({"source": src, "out": out, "original_sec": original_sec, "cut_sec": cut_sec})

print("=" * 60)
print("NEW يلا VARIANTS INTO comparison4/")
print("=" * 60)

voice = ensure_edge_voice_for_round4()
print("Selected voice:", voice)

async def generate_yalla_round4():
    COMPARISON4_YALLA_LOG.clear()
    for idx, variant in enumerate(YALLA_VARIANTS_ROUND4, start=1):
        tts_text = variant.strip()
        if not tts_text.endswith((".", "!", "?")):
            tts_text += "."
        out = DIR_COMPARISON4 / f"word_يلا_v{idx:02d}.mp3"
        await synth_round4_mp3(tts_text, out, voice)
        print("original word: يلا")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output path: {out.relative_to(BASE_DIR)}")
        print("---")
        COMPARISON4_YALLA_LOG.append({"variant": variant, "tts_text": tts_text, "out": out})

asyncio.get_event_loop().run_until_complete(generate_yalla_round4())

print("=" * 60)
print("ROUND 4 SUMMARY")
print("=" * 60)
print("files moved to chosen:", len(COMPARISON4_MOVED_TO_CHOSEN))
for item in COMPARISON4_MOVED_TO_CHOSEN:
    print(" -", item["file"])
print("recut files created:", len(COMPARISON4_RECUT_LOG))
for item in COMPARISON4_RECUT_LOG:
    print(" -", item["out"].relative_to(BASE_DIR))
print("يلا variants generated:", len(COMPARISON4_YALLA_LOG))
for item in COMPARISON4_YALLA_LOG:
    print(" -", item["out"].relative_to(BASE_DIR))
print("missing files:", len(COMPARISON4_MISSING))
for item in COMPARISON4_MISSING:
    print(" -", item)

chosen_checks = [DIR_CHOSEN / _mp3_name(stem) for stem in COMPARISON4_MOVE_TO_CHOSEN]
cut_checks = [DIR_COMPARISON4_CUT / out_name for _, _, out_name in COMPARISON4_RECUT_JOBS]
yalla_checks = [DIR_COMPARISON4 / f"word_يلا_v{idx:02d}.mp3" for idx in range(1, len(YALLA_VARIANTS_ROUND4) + 1)]
print("chosen files exist:", all(path.is_file() for path in chosen_checks))
print("comparison4/cut files exist:", all(path.is_file() for path in cut_checks))
print("comparison4 يلا variants exist:", all(path.is_file() for path in yalla_checks))

MOVE APPROVED CUTS TO chosen/
moved to chosen: word_روح_v01_cut_0_6.mp3
source folder: generated_audio_words\edge_tts_words\comparison3\cut
destination folder: generated_audio_words\edge_tts_words\chosen
removed duplicate: generated_audio_words\edge_tts_words\comparison2\cut\word_روح_v01_cut_0_6.mp3
moved to chosen: word_تمام_v02_cut_0_6.mp3
source folder: generated_audio_words\edge_tts_words\comparison3\cut
destination folder: generated_audio_words\edge_tts_words\chosen
RECUT FILES INTO comparison4/cut/
source: generated_audio_words\edge_tts_words\comparison2\word_معايا_v01.mp3
original duration: 1.752s
new cut duration: 0.650s
saved path: generated_audio_words\edge_tts_words\comparison4\cut\word_معايا_v01_cut_0_65.mp3
---


source: generated_audio_words\edge_tts_words\comparison2\word_كويس_v01.mp3
original duration: 1.656s
new cut duration: 0.650s
saved path: generated_audio_words\edge_tts_words\comparison4\cut\word_كويس_v01_cut_0_65.mp3
---
source: generated_audio_words\edge_tts_words\comparison2\word_شكرا_v05.mp3
original duration: 1.704s
new cut duration: 0.650s
saved path: generated_audio_words\edge_tts_words\comparison4\cut\word_شكرا_v05_cut_0_65.mp3
---


source: generated_audio_words\edge_tts_words\comparison\word_لو_سمحت_v01.mp3
original duration: 1.944s
new cut duration: 0.800s
saved path: generated_audio_words\edge_tts_words\comparison4\cut\word_لو_سمحت_v01_cut_0_8.mp3
---
NEW يلا VARIANTS INTO comparison4/


Selected voice: ar-EG-SalmaNeural


original word: يلا
variant text sent to Edge TTS: يَلَّا.
output path: generated_audio_words\edge_tts_words\comparison4\word_يلا_v01.mp3
---


...

original word: يلا
variant text sent to Edge TTS: yalla.
output path: generated_audio_words\edge_tts_words\comparison4\word_يلا_v06.mp3
---


original word: يلا
variant text sent to Edge TTS: yallaa.
output path: generated_audio_words\edge_tts_words\comparison4\word_يلا_v07.mp3
---


original word: يلا
variant text sent to Edge TTS: yal-la.
output path: generated_audio_words\edge_tts_words\comparison4\word_يلا_v08.mp3
---


original word: يلا
variant text sent to Edge TTS: ya lla.
output path: generated_audio_words\edge_tts_words\comparison4\word_يلا_v09.mp3
---


original word: يلا
variant text sent to Edge TTS: yalla!
output path: generated_audio_words\edge_tts_words\comparison4\word_يلا_v10.mp3
---
ROUND 4 SUMMARY
files moved to chosen: 2
 - word_روح_v01_cut_0_6.mp3
 - word_تمام_v02_cut_0_6.mp3
recut files created: 4
 - generated_audio_words\edge_tts_words\comparison4\cut\word_معايا_v01_cut_0_65.mp3
 - generated_audio_words\edge_tts_words\comparison4\cut\word_كويس_v01_cut_0_65.mp3
 - generated_audio_words\edge_tts_words\comparison4\cut\word_شكرا_v05_cut_0_65.mp3
 - generated_audio_words\edge_tts_words\comparison4\cut\word_لو_سمحت_v01_cut_0_8.mp3
يلا variants generated: 10
 - generated_audio_words\edge_tts_words\comparison4\word_يلا_v01.mp3
 - generated_audio_words\edge_tts_words\comparison4\word_يلا_v02.mp3
 - generated_audio_words\edge_tts_words\comparison4\word_يلا_v03.mp3
 - generated_audio_words\edge_tts_words\comparison4\word_يلا_v04.mp3
 - generated_audio_words\edge_tts_words\comparison4\word_يلا_v05.mp3
 - generated_audio_words\edge_tt

## Round 4 Playback

Display playback widgets for the new 0.65s recuts, the new 0.8s recut, and the new `يلا` variants.

In [3]:
if COMPARISON4_RECUT_LOG:
    print("=" * 60)
    print("comparison4/cut recuts")
    print("=" * 60)
    for item in COMPARISON4_RECUT_LOG:
        print(f"{item['out'].name} | {item['original_sec']:.3f}s -> {item['cut_sec']:.3f}s")
        display(IPA(filename=str(item["out"]), autoplay=False))

if COMPARISON4_YALLA_LOG:
    print("=" * 60)
    print("comparison4 يلا variants")
    print("=" * 60)
    for item in COMPARISON4_YALLA_LOG:
        print(f"{item['out'].name} | TTS: {item['tts_text']}")
        display(IPA(filename=str(item["out"]), autoplay=False))

print("=" * 60)
print("chosen moved cut examples")
print("=" * 60)
for stem in COMPARISON4_MOVE_TO_CHOSEN:
    path = DIR_CHOSEN / _mp3_name(stem)
    print(f"{path.relative_to(BASE_DIR)} | exists: {path.is_file()}")
    if path.is_file():
        display(IPA(filename=str(path), autoplay=False))

comparison4/cut recuts
word_معايا_v01_cut_0_65.mp3 | 1.752s -> 0.650s


word_كويس_v01_cut_0_65.mp3 | 1.656s -> 0.650s


word_شكرا_v05_cut_0_65.mp3 | 1.704s -> 0.650s


word_لو_سمحت_v01_cut_0_8.mp3 | 1.944s -> 0.800s


comparison4 يلا variants
word_يلا_v01.mp3 | TTS: يَلَّا.


...

word_يلا_v08.mp3 | TTS: yal-la.


word_يلا_v09.mp3 | TTS: ya lla.


word_يلا_v10.mp3 | TTS: yalla!


chosen moved cut examples
generated_audio_words\edge_tts_words\chosen\word_روح_v01_cut_0_6.mp3 | exists: True


generated_audio_words\edge_tts_words\chosen\word_تمام_v02_cut_0_6.mp3 | exists: True


## Round 5 — comparison5 (recuts and يلا)

Use `generated_audio_words/` as the base folder.

- Recut selected clips into `comparison5/cut/` without overwriting previous cuts
- Move `word_كويس_v01_cut_0_65.mp3` into `chosen/`
- Generate a new set of `يلا` variants in `comparison5/`, starting with the original unmapped `يلا.`

In [2]:
DIR_COMPARISON5.mkdir(parents=True, exist_ok=True)
DIR_COMPARISON5_CUT.mkdir(parents=True, exist_ok=True)
DIR_CHOSEN.mkdir(parents=True, exist_ok=True)

_ensure("pydub")
from pydub import AudioSegment

COMPARISON5_RECUT_JOBS = [
    ("word_شكرا_v05", 0.67, "word_شكرا_v05_cut_0_67.mp3"),
    ("word_لو_سمحت_v01", 0.85, "word_لو_سمحت_v01_cut_0_85.mp3"),
    ("word_معايا_v01", 0.70, "word_معايا_v01_cut_0_7.mp3"),
]

COMPARISON5_MOVE_TO_CHOSEN = [
    "word_كويس_v01_cut_0_65",
]

YALLA_VARIANTS_ROUND5 = [
    "يلا.",      # original word, no mapping
    "يَلّا.",
    "يَلَّا.",
    "يَالَا.",
    "يَالا.",
    "ياه لا.",
    "يا لا.",
    "يَلا.",
    "YAH-lah.",
    "Yalla.",
    "yalla.",
    "yah-lah.",
    "ya-lah.",
    "yallah.",
    "ya lla.",
    "yal lah.",
    "يَل لا.",
]

COMPARISON5_CUT_SOURCE_DIRS = [DIR_COMPARISON4_CUT, DIR_COMPARISON3_CUT, DIR_COMPARISON2_CUT, DIR_CUT]
COMPARISON5_ORIGINAL_SOURCE_DIRS = [DIR_COMPARISON2, DIR_COMPARISON, DIR_COMPARISON3, DIR_COMPARISON4, DIR_CHOSEN]


def _mp3_name_round5(stem: str) -> str:
    return stem if stem.endswith(".mp3") else f"{stem}.mp3"


def find_cut_source_round5(stem: str) -> Path | None:
    name = _mp3_name_round5(stem)
    for folder in COMPARISON5_CUT_SOURCE_DIRS:
        candidate = folder / name
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    chosen_candidate = DIR_CHOSEN / name
    if chosen_candidate.is_file() and chosen_candidate.stat().st_size > 0:
        return chosen_candidate
    return None


def find_original_source_round5(stem: str) -> Path | None:
    name = _mp3_name_round5(stem)
    for folder in COMPARISON5_ORIGINAL_SOURCE_DIRS:
        candidate = folder / name
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    return None


def ensure_edge_voice_for_round5() -> str:
    global EDGE_VOICE
    if "EDGE_VOICE" in globals() and EDGE_VOICE:
        return EDGE_VOICE

    async def _select_voice() -> str:
        voices = await edge_tts.list_voices()
        by_short = {v["ShortName"]: v for v in voices}
        for pref in ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]:
            if pref in by_short:
                return pref
        raise RuntimeError("No Egyptian Arabic Edge TTS voice found")

    EDGE_VOICE = asyncio.get_event_loop().run_until_complete(_select_voice())
    return EDGE_VOICE


async def synth_round5_mp3(text: str, mp3_path: Path, voice: str) -> float:
    mp3_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    last_exc = None
    for attempt in range(4):
        try:
            comm = edge_tts.Communicate(text, voice)
            await comm.save(str(mp3_path))
            if mp3_path.is_file() and mp3_path.stat().st_size > 0:
                return time.perf_counter() - t0
        except Exception as exc:
            last_exc = exc
            if mp3_path.is_file() and mp3_path.stat().st_size == 0:
                mp3_path.unlink(missing_ok=True)
            await asyncio.sleep(1.5 * (attempt + 1))
    if last_exc:
        raise last_exc
    raise RuntimeError("synthesis produced empty file")


COMPARISON5_MOVED_TO_CHOSEN: list[dict] = []
COMPARISON5_RECUT_LOG: list[dict] = []
COMPARISON5_YALLA_LOG: list[dict] = []
COMPARISON5_MISSING: list[str] = []

print("=" * 60)
print("MOVE APPROVED CUT TO chosen/")
print("=" * 60)

for stem in COMPARISON5_MOVE_TO_CHOSEN:
    filename = _mp3_name_round5(stem)
    dest = DIR_CHOSEN / filename
    src = find_cut_source_round5(stem)

    if src is None:
        if dest.is_file():
            print(f"already in chosen: {dest.relative_to(BASE_DIR)}")
            COMPARISON5_MOVED_TO_CHOSEN.append({"file": filename, "source": "already in chosen", "destination": dest})
        else:
            print(f"MISSING move source: {filename}")
            COMPARISON5_MISSING.append(filename)
        continue

    if src.resolve() != dest.resolve():
        if dest.is_file():
            dest.unlink()
        shutil.move(str(src), str(dest))
        print(f"file moved to chosen: {filename}")
        print(f"source folder: {src.parent.relative_to(BASE_DIR)}")
        print(f"destination folder: {DIR_CHOSEN.relative_to(BASE_DIR)}")
    else:
        print(f"already in chosen: {dest.relative_to(BASE_DIR)}")

    for folder in COMPARISON5_CUT_SOURCE_DIRS:
        duplicate = folder / filename
        if duplicate.is_file() and duplicate.resolve() != dest.resolve():
            duplicate.unlink()
            print(f"removed duplicate: {duplicate.relative_to(BASE_DIR)}")

    COMPARISON5_MOVED_TO_CHOSEN.append({"file": filename, "source": src.parent, "destination": dest})

print("=" * 60)
print("RECUT FILES INTO comparison5/cut/")
print("=" * 60)

for source_stem, target_sec, out_name in COMPARISON5_RECUT_JOBS:
    src = find_original_source_round5(source_stem)
    if src is None:
        print(f"MISSING recut source: {source_stem}.mp3")
        COMPARISON5_MISSING.append(f"{source_stem}.mp3")
        continue

    audio = AudioSegment.from_mp3(src)
    original_sec = len(audio) / 1000.0
    cut_audio = audio[: int(target_sec * 1000)]
    cut_sec = len(cut_audio) / 1000.0
    out = DIR_COMPARISON5_CUT / out_name
    cut_audio.export(out, format="mp3")

    print(f"source: {src.relative_to(BASE_DIR)}")
    print(f"original duration: {original_sec:.3f}s")
    print(f"new cut duration: {cut_sec:.3f}s")
    print(f"saved path: {out.relative_to(BASE_DIR)}")
    print("---")
    COMPARISON5_RECUT_LOG.append({"source": src, "out": out, "original_sec": original_sec, "cut_sec": cut_sec})

print("=" * 60)
print("NEW يلا VARIANTS INTO comparison5/")
print("=" * 60)

voice = ensure_edge_voice_for_round5()
print("Selected voice:", voice)

async def generate_yalla_round5():
    COMPARISON5_YALLA_LOG.clear()
    for idx, variant in enumerate(YALLA_VARIANTS_ROUND5, start=1):
        tts_text = variant.strip()
        if not tts_text.endswith((".", "!", "?")):
            tts_text += "."
        out = DIR_COMPARISON5 / f"word_يلا_v{idx:02d}.mp3"
        await synth_round5_mp3(tts_text, out, voice)
        print("original word: يلا")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"يلا output path: {out.relative_to(BASE_DIR)}")
        print("---")
        COMPARISON5_YALLA_LOG.append({"variant": variant, "tts_text": tts_text, "out": out})

asyncio.get_event_loop().run_until_complete(generate_yalla_round5())

print("=" * 60)
print("ROUND 5 SUMMARY")
print("=" * 60)
print("file moved to chosen:", len(COMPARISON5_MOVED_TO_CHOSEN))
for item in COMPARISON5_MOVED_TO_CHOSEN:
    print(" -", item["file"])
print("recut files created:", len(COMPARISON5_RECUT_LOG))
for item in COMPARISON5_RECUT_LOG:
    print(" -", item["out"].relative_to(BASE_DIR))
print("يلا variants generated:", len(COMPARISON5_YALLA_LOG))
for item in COMPARISON5_YALLA_LOG:
    print(" -", item["out"].relative_to(BASE_DIR))
print("missing files:", len(COMPARISON5_MISSING))
for item in COMPARISON5_MISSING:
    print(" -", item)

chosen_checks = [DIR_CHOSEN / _mp3_name_round5(stem) for stem in COMPARISON5_MOVE_TO_CHOSEN]
cut_checks = [DIR_COMPARISON5_CUT / out_name for _, _, out_name in COMPARISON5_RECUT_JOBS]
yalla_checks = [DIR_COMPARISON5 / f"word_يلا_v{idx:02d}.mp3" for idx in range(1, len(YALLA_VARIANTS_ROUND5) + 1)]
print("chosen كويس cut exists:", all(path.is_file() for path in chosen_checks))
print("comparison5/cut files exist:", all(path.is_file() for path in cut_checks))
print("comparison5 يلا variants exist:", all(path.is_file() for path in yalla_checks))

MOVE APPROVED CUT TO chosen/
file moved to chosen: word_كويس_v01_cut_0_65.mp3
source folder: generated_audio_words\edge_tts_words\comparison4\cut
destination folder: generated_audio_words\edge_tts_words\chosen
RECUT FILES INTO comparison5/cut/
source: generated_audio_words\edge_tts_words\comparison2\word_شكرا_v05.mp3
original duration: 1.704s
new cut duration: 0.670s
saved path: generated_audio_words\edge_tts_words\comparison5\cut\word_شكرا_v05_cut_0_67.mp3
---


source: generated_audio_words\edge_tts_words\comparison\word_لو_سمحت_v01.mp3
original duration: 1.944s
new cut duration: 0.850s
saved path: generated_audio_words\edge_tts_words\comparison5\cut\word_لو_سمحت_v01_cut_0_85.mp3
---
source: generated_audio_words\edge_tts_words\comparison2\word_معايا_v01.mp3
original duration: 1.752s
new cut duration: 0.700s
saved path: generated_audio_words\edge_tts_words\comparison5\cut\word_معايا_v01_cut_0_7.mp3
---
NEW يلا VARIANTS INTO comparison5/


Selected voice: ar-EG-SalmaNeural


original word: يلا
variant text sent to Edge TTS: يلا.
يلا output path: generated_audio_words\edge_tts_words\comparison5\word_يلا_v01.mp3
---


original word: يلا
variant text sent to Edge TTS: يَلّا.
يلا output path: generated_audio_words\edge_tts_words\comparison5\word_يلا_v02.mp3
---


...

original word: يلا
variant text sent to Edge TTS: ya-lah.
يلا output path: generated_audio_words\edge_tts_words\comparison5\word_يلا_v13.mp3
---


original word: يلا
variant text sent to Edge TTS: yallah.
يلا output path: generated_audio_words\edge_tts_words\comparison5\word_يلا_v14.mp3
---


original word: يلا
variant text sent to Edge TTS: ya lla.
يلا output path: generated_audio_words\edge_tts_words\comparison5\word_يلا_v15.mp3
---


original word: يلا
variant text sent to Edge TTS: yal lah.
يلا output path: generated_audio_words\edge_tts_words\comparison5\word_يلا_v16.mp3
---


original word: يلا
variant text sent to Edge TTS: يَل لا.
يلا output path: generated_audio_words\edge_tts_words\comparison5\word_يلا_v17.mp3
---
ROUND 5 SUMMARY
file moved to chosen: 1
 - word_كويس_v01_cut_0_65.mp3
recut files created: 3
 - generated_audio_words\edge_tts_words\comparison5\cut\word_شكرا_v05_cut_0_67.mp3
 - generated_audio_words\edge_tts_words\comparison5\cut\word_لو_سمحت_v01_cut_0_85.mp3
 - generated_audio_words\edge_tts_words\comparison5\cut\word_معايا_v01_cut_0_7.mp3
يلا variants generated: 17
 - generated_audio_words\edge_tts_words\comparison5\word_يلا_v01.mp3
 - generated_audio_words\edge_tts_words\comparison5\word_يلا_v02.mp3
 - generated_audio_words\edge_tts_words\comparison5\word_يلا_v03.mp3
 - generated_audio_words\edge_tts_words\comparison5\word_يلا_v04.mp3
 - generated_audio_words\edge_tts_words\comparison5\word_يلا_v05.mp3
 - generated_audio_words\edge_tts_words\comparison5\word_يلا_v06.mp3
 - generated_audio_words\edge_tts_words\comparison5\word_يلا_v07.mp3


## Round 5 Playback

Display playback widgets for the new comparison5 cut files and all new `يلا` variants.

In [3]:
if COMPARISON5_RECUT_LOG:
    print("=" * 60)
    print("comparison5/cut recuts")
    print("=" * 60)
    for item in COMPARISON5_RECUT_LOG:
        print(f"{item['out'].name} | {item['original_sec']:.3f}s -> {item['cut_sec']:.3f}s")
        display(IPA(filename=str(item["out"]), autoplay=False))

if COMPARISON5_YALLA_LOG:
    print("=" * 60)
    print("comparison5 يلا variants")
    print("=" * 60)
    for item in COMPARISON5_YALLA_LOG:
        print(f"{item['out'].name} | TTS: {item['tts_text']}")
        display(IPA(filename=str(item["out"]), autoplay=False))

print("=" * 60)
print("chosen كويس cut")
print("=" * 60)
for stem in COMPARISON5_MOVE_TO_CHOSEN:
    path = DIR_CHOSEN / _mp3_name_round5(stem)
    print(f"{path.relative_to(BASE_DIR)} | exists: {path.is_file()}")
    if path.is_file():
        display(IPA(filename=str(path), autoplay=False))

comparison5/cut recuts
word_شكرا_v05_cut_0_67.mp3 | 1.704s -> 0.670s


word_لو_سمحت_v01_cut_0_85.mp3 | 1.944s -> 0.850s


word_معايا_v01_cut_0_7.mp3 | 1.752s -> 0.700s


comparison5 يلا variants
word_يلا_v01.mp3 | TTS: يلا.


word_يلا_v02.mp3 | TTS: يَلّا.


...

word_يلا_v14.mp3 | TTS: yallah.


word_يلا_v15.mp3 | TTS: ya lla.


word_يلا_v16.mp3 | TTS: yal lah.


word_يلا_v17.mp3 | TTS: يَل لا.


chosen كويس cut
generated_audio_words\edge_tts_words\chosen\word_كويس_v01_cut_0_65.mp3 | exists: True


## Move Final Approved Round 5 Files to chosen/

Use `generated_audio_words/edge_tts_words/chosen/`.

Move the approved Round 5 files into `chosen/` without overwriting an existing chosen file. After each file is safely present in `chosen/`, remove duplicate copies from comparison folders.

In [2]:
FINAL_ROUND5_CHOSEN_STEMS = [
    "word_يلا_v03",
    "word_شكرا_v05_cut_0_67",
    "word_لو_سمحت_v01_cut_0_85",
    "word_معايا_v01_cut_0_7",
]

FINAL_ROUND5_SOURCE_DIRS = [
    DIR_COMPARISON5,
    DIR_COMPARISON5_CUT,
    DIR_COMPARISON4,
    DIR_COMPARISON4_CUT,
    DIR_COMPARISON3,
    DIR_COMPARISON3_CUT,
    DIR_COMPARISON2,
    DIR_COMPARISON2_CUT,
    DIR_COMPARISON,
    DIR_CUT,
]

DIR_CHOSEN.mkdir(parents=True, exist_ok=True)

FINAL_ROUND5_MOVED: list[dict] = []
FINAL_ROUND5_ALREADY_CHOSEN: list[str] = []
FINAL_ROUND5_REMOVED_DUPLICATES: list[dict] = []
FINAL_ROUND5_MISSING: list[str] = []

print("chosen folder:", DIR_CHOSEN.relative_to(BASE_DIR))
print("source folders:")
for folder in FINAL_ROUND5_SOURCE_DIRS:
    print(" -", folder.relative_to(BASE_DIR))
print("=" * 60)

for stem in FINAL_ROUND5_CHOSEN_STEMS:
    filename = f"{stem}.mp3"
    dest = DIR_CHOSEN / filename
    source = None

    if dest.is_file() and dest.stat().st_size > 0:
        FINAL_ROUND5_ALREADY_CHOSEN.append(filename)
        print(f"already in chosen (not overwritten): {dest.relative_to(BASE_DIR)}")
    else:
        for folder in FINAL_ROUND5_SOURCE_DIRS:
            candidate = folder / filename
            if candidate.is_file() and candidate.stat().st_size > 0:
                source = candidate
                break

        if source is None:
            FINAL_ROUND5_MISSING.append(filename)
            print(f"MISSING: {filename}")
            continue

        shutil.move(str(source), str(dest))
        FINAL_ROUND5_MOVED.append({
            "file": filename,
            "source": source.parent.relative_to(BASE_DIR),
            "destination": DIR_CHOSEN.relative_to(BASE_DIR),
        })
        print(f"MOVED: {filename}")
        print(f"  source folder: {source.parent.relative_to(BASE_DIR)}")
        print(f"  destination folder: {DIR_CHOSEN.relative_to(BASE_DIR)}")

    if dest.is_file() and dest.stat().st_size > 0:
        for folder in FINAL_ROUND5_SOURCE_DIRS:
            duplicate = folder / filename
            if duplicate.is_file():
                duplicate.unlink()
                FINAL_ROUND5_REMOVED_DUPLICATES.append({
                    "file": filename,
                    "removed_from": folder.relative_to(BASE_DIR),
                })
                print(f"REMOVED duplicate: {duplicate.relative_to(BASE_DIR)}")

print("=" * 60)
print("files moved:", len(FINAL_ROUND5_MOVED))
for item in FINAL_ROUND5_MOVED:
    print(f" - {item['file']} | {item['source']} -> {item['destination']}")
print("already in chosen:", len(FINAL_ROUND5_ALREADY_CHOSEN))
for filename in FINAL_ROUND5_ALREADY_CHOSEN:
    print(f" - {filename}")
print("duplicates removed:", len(FINAL_ROUND5_REMOVED_DUPLICATES))
for item in FINAL_ROUND5_REMOVED_DUPLICATES:
    print(f" - {item['file']} from {item['removed_from']}")
print("missing files:", len(FINAL_ROUND5_MISSING))
for filename in FINAL_ROUND5_MISSING:
    print(f" - {filename}")

chosen_checks = [DIR_CHOSEN / f"{stem}.mp3" for stem in FINAL_ROUND5_CHOSEN_STEMS]
comparison_duplicates = []
for stem in FINAL_ROUND5_CHOSEN_STEMS:
    filename = f"{stem}.mp3"
    for folder in FINAL_ROUND5_SOURCE_DIRS:
        duplicate = folder / filename
        if duplicate.is_file():
            comparison_duplicates.append(duplicate)

print("all moved files exist in chosen:", all(path.is_file() for path in chosen_checks))
print("old comparison copies remaining:", len(comparison_duplicates))
if comparison_duplicates:
    for path in comparison_duplicates:
        print(" -", path.relative_to(BASE_DIR))

chosen folder: generated_audio_words\edge_tts_words\chosen
source folders:
 - generated_audio_words\edge_tts_words\comparison5
 - generated_audio_words\edge_tts_words\comparison5\cut
 - generated_audio_words\edge_tts_words\comparison4
 - generated_audio_words\edge_tts_words\comparison4\cut
 - generated_audio_words\edge_tts_words\comparison3
 - generated_audio_words\edge_tts_words\comparison3\cut
 - generated_audio_words\edge_tts_words\comparison2
 - generated_audio_words\edge_tts_words\comparison2\cut
 - generated_audio_words\edge_tts_words\comparison
 - generated_audio_words\edge_tts_words\cut
MOVED: word_يلا_v03.mp3
  source folder: generated_audio_words\edge_tts_words\comparison5
  destination folder: generated_audio_words\edge_tts_words\chosen
REMOVED duplicate: generated_audio_words\edge_tts_words\comparison4\word_يلا_v03.mp3
REMOVED duplicate: generated_audio_words\edge_tts_words\comparison3\word_يلا_v03.mp3
REMOVED duplicate: generated_audio_words\edge_tts_words\comparison2\word

## Final Round 5 Chosen Playback

Display playback widgets for the moved chosen files.

In [3]:
print("Moved chosen playback:")
for stem in FINAL_ROUND5_CHOSEN_STEMS:
    path = DIR_CHOSEN / f"{stem}.mp3"
    print(f" - {path.relative_to(BASE_DIR)} | exists: {path.is_file()}")
    if path.is_file():
        display(IPA(filename=str(path), autoplay=False))

Moved chosen playback:
 - generated_audio_words\edge_tts_words\chosen\word_يلا_v03.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_شكرا_v05_cut_0_67.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_لو_سمحت_v01_cut_0_85.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_معايا_v01_cut_0_7.mp3 | exists: True


## Round 6 — comparison6 (additional Egyptian word variants)

Use `generated_audio_words/` as the base folder.

Generate Egyptian Arabic pronunciation variants for the next set of words into:

`generated_audio_words/edge_tts_words/comparison6/`

MP3 only. No WAV generation and no audio cleaning.

In [2]:
DIR_COMPARISON6.mkdir(parents=True, exist_ok=True)

ROUND6_WORD_VARIANTS: dict[str, list[str]] = {
    "نتعرف": [
        "نِتْعَرَف.",
        "نِتعارف.",
        "net3araf.",
        "نِتْعَرِف.",
    ],
    "أنا": [
        "أَنَا.",
        "أَنَاا.",
        "ana.",
        "أَنَ.",
    ],
    "أسمي": [
        "اِسْمِي.",
        "إِسْمِي.",
        "ismii.",
        "اِسمي.",
    ],
    "عليكم": [
        "عَلَيْكُم.",
        "عَلِيكُم.",
        "3alekom.",
        "عَلِيكوُم.",
    ],
    "رحمة": [
        "رَحْمَة.",
        "رَحمه.",
        "ra7ma.",
        "رَحْمَه.",
    ],
    "الله": [
        "الله.",
        "اللّه.",
        "allah.",
        "أللاّه.",
    ],
    "بركاته": [
        "بَرَكَاتُه.",
        "بَرَكاتُه.",
        "barakato.",
        "بَرَكاتو.",
    ],
    "أهلا": [
        "أَهْلَا.",
        "أَهلا.",
        "ahlan.",
        "أَهْلَاا.",
    ],
    "اتفضل": [
        "اِتْفَضَّل.",
        "اتفضل.",
        "etfaddal.",
        "اِتْفَضَل.",
    ],
    "سنك": [
        "سِنَّك.",
        "سِنِك.",
        "sennak.",
        "سِنك.",
    ],
    "عاوز": [
        "عَاوِز.",
        "عاوِز.",
        "3awiz.",
        "عَاوْز.",
    ],
    "بحب": [
        "بَحِب.",
        "بَحَب.",
        "ba7eb.",
        "بَحِبْ.",
    ],
    "وعد": [
        "وَعْد.",
        "وَعَد.",
        "wa3d.",
        "وَعْدْ.",
    ],
    "مهذب": [
        "مُهَذَّب.",
        "مُهَزَّب.",
        "mohazzab.",
        "مُهَذِب.",
    ],
    "صديق": [
        "صَدِيق.",
        "صَاحِب.",
        "sadee2.",
        "صَدِيقْ.",
    ],
}

ROUND6_DIACRITICS = "ًٌٍَُِّْـ"


def round6_word_slug(word: str) -> str:
    slug = word.strip().replace(" ", "_")
    for mark in ROUND6_DIACRITICS:
        slug = slug.replace(mark, "")
    return slug


def ensure_edge_voice_for_round6() -> str:
    global EDGE_VOICE
    if "EDGE_VOICE" in globals() and EDGE_VOICE:
        return EDGE_VOICE

    async def _select_voice() -> str:
        voices = await edge_tts.list_voices()
        by_short = {v["ShortName"]: v for v in voices}
        for pref in ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]:
            if pref in by_short:
                return pref
        raise RuntimeError("No Egyptian Arabic Edge TTS voice found")

    EDGE_VOICE = asyncio.get_event_loop().run_until_complete(_select_voice())
    return EDGE_VOICE


async def synth_round6_mp3(text: str, mp3_path: Path, voice: str) -> float:
    mp3_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    last_exc = None
    for attempt in range(4):
        try:
            comm = edge_tts.Communicate(text, voice)
            await comm.save(str(mp3_path))
            if mp3_path.is_file() and mp3_path.stat().st_size > 0:
                return time.perf_counter() - t0
        except Exception as exc:
            last_exc = exc
            if mp3_path.is_file() and mp3_path.stat().st_size == 0:
                mp3_path.unlink(missing_ok=True)
            await asyncio.sleep(1.5 * (attempt + 1))
    if last_exc:
        raise last_exc
    raise RuntimeError("synthesis produced empty file")


COMPARISON6_LOG: list[dict] = []
COMPARISON6_MISSING: list[Path] = []

voice = ensure_edge_voice_for_round6()
print("Selected voice:", voice)
print("comparison6 folder:", DIR_COMPARISON6.relative_to(BASE_DIR))
print("words:", len(ROUND6_WORD_VARIANTS))
print("variants to generate:", sum(len(v) for v in ROUND6_WORD_VARIANTS.values()))

async def generate_round6_variants():
    COMPARISON6_LOG.clear()
    for word, variants in ROUND6_WORD_VARIANTS.items():
        for idx, variant in enumerate(variants, start=1):
            tts_text = variant.strip()
            if not tts_text.endswith((".", "!", "?")):
                tts_text += "."
            out = DIR_COMPARISON6 / f"word_{round6_word_slug(word)}_v{idx:02d}.mp3"
            await synth_round6_mp3(tts_text, out, voice)
            print(f"original word: {word}")
            print(f"variant text sent to Edge TTS: {tts_text}")
            print(f"output path: {out.relative_to(BASE_DIR)}")
            print("---")
            COMPARISON6_LOG.append({
                "word": word,
                "variant": variant,
                "tts_text": tts_text,
                "out": out,
            })

asyncio.get_event_loop().run_until_complete(generate_round6_variants())

expected_paths = [
    DIR_COMPARISON6 / f"word_{round6_word_slug(word)}_v{idx:02d}.mp3"
    for word, variants in ROUND6_WORD_VARIANTS.items()
    for idx in range(1, len(variants) + 1)
]
COMPARISON6_MISSING = [path for path in expected_paths if not path.is_file() or path.stat().st_size == 0]

print("=" * 60)
print("ROUND 6 SUMMARY")
print("=" * 60)
print("generated comparison6 MP3 files:", len(COMPARISON6_LOG))
print("expected comparison6 MP3 files:", len(expected_paths))
print("missing comparison6 MP3 files:", len(COMPARISON6_MISSING))
for path in COMPARISON6_MISSING:
    print(" -", path.relative_to(BASE_DIR))
print("comparison6 files exist:", len(COMPARISON6_MISSING) == 0)

Selected voice: ar-EG-SalmaNeural
comparison6 folder: generated_audio_words\edge_tts_words\comparison6
words: 15
variants to generate: 60


original word: نتعرف
variant text sent to Edge TTS: نِتْعَرَف.
output path: generated_audio_words\edge_tts_words\comparison6\word_نتعرف_v01.mp3
---


original word: نتعرف
variant text sent to Edge TTS: نِتعارف.
output path: generated_audio_words\edge_tts_words\comparison6\word_نتعرف_v02.mp3
---


original word: نتعرف
variant text sent to Edge TTS: net3araf.
output path: generated_audio_words\edge_tts_words\comparison6\word_نتعرف_v03.mp3
---


original word: نتعرف
variant text sent to Edge TTS: نِتْعَرِف.
output path: generated_audio_words\edge_tts_words\comparison6\word_نتعرف_v04.mp3
---


...

original word: مهذب
variant text sent to Edge TTS: مُهَذِب.
output path: generated_audio_words\edge_tts_words\comparison6\word_مهذب_v04.mp3
---


original word: صديق
variant text sent to Edge TTS: صَدِيق.
output path: generated_audio_words\edge_tts_words\comparison6\word_صديق_v01.mp3
---


original word: صديق
variant text sent to Edge TTS: صَاحِب.
output path: generated_audio_words\edge_tts_words\comparison6\word_صديق_v02.mp3
---


original word: صديق
variant text sent to Edge TTS: sadee2.
output path: generated_audio_words\edge_tts_words\comparison6\word_صديق_v03.mp3
---


original word: صديق
variant text sent to Edge TTS: صَدِيقْ.
output path: generated_audio_words\edge_tts_words\comparison6\word_صديق_v04.mp3
---
ROUND 6 SUMMARY
generated comparison6 MP3 files: 60
expected comparison6 MP3 files: 60
missing comparison6 MP3 files: 0
comparison6 files exist: True


## Round 6 Playback

Display playback widgets grouped by word for the new comparison6 variants.

In [3]:
from collections import defaultdict

comparison6_by_word: dict[str, list[dict]] = defaultdict(list)
for entry in COMPARISON6_LOG:
    comparison6_by_word[entry["word"]].append(entry)

for word in ROUND6_WORD_VARIANTS:
    entries = comparison6_by_word.get(word, [])
    if not entries:
        continue
    print("=" * 60)
    print(f"ROUND 6 — WORD: {word} ({len(entries)} variants)")
    print("=" * 60)
    for entry in entries:
        print(f"{entry['out'].name} | TTS: {entry['tts_text']}")
        display(IPA(filename=str(entry["out"]), autoplay=False))

ROUND 6 — WORD: نتعرف (4 variants)
word_نتعرف_v01.mp3 | TTS: نِتْعَرَف.


word_نتعرف_v02.mp3 | TTS: نِتعارف.


word_نتعرف_v03.mp3 | TTS: net3araf.


word_نتعرف_v04.mp3 | TTS: نِتْعَرِف.


ROUND 6 — WORD: أنا (4 variants)
word_أنا_v01.mp3 | TTS: أَنَا.


...

word_مهذب_v04.mp3 | TTS: مُهَذِب.


ROUND 6 — WORD: صديق (4 variants)
word_صديق_v01.mp3 | TTS: صَدِيق.


word_صديق_v02.mp3 | TTS: صَاحِب.


word_صديق_v03.mp3 | TTS: sadee2.


word_صديق_v04.mp3 | TTS: صَدِيقْ.


## Round 7 — comparison7 moves and cut

Use `generated_audio_words/` as the base folder.

- Move the approved Round 6 word variants into `chosen/`
- Remove those moved files from comparison folders after they are safely in `chosen/`
- Cut `word_الله_v02.mp3` to ~0.6s into `comparison7/cut/` without overwriting the original

In [2]:
DIR_COMPARISON7.mkdir(parents=True, exist_ok=True)
DIR_COMPARISON7_CUT.mkdir(parents=True, exist_ok=True)
DIR_CHOSEN.mkdir(parents=True, exist_ok=True)

_ensure("pydub")
from pydub import AudioSegment

ROUND7_MOVE_TO_CHOSEN_STEMS = [
    "word_اتفضل_v01",
    "word_أسمي_v01",
    "word_أنا_v01",
    "word_أهلا_v02",
    "word_بحب_v01",
    "word_بركاته_v04",
    "word_رحمة_v01",
    "word_سنك_v01",
    "word_صديق_v04",
    "word_عاوز_v04",
    "word_عليكم_v02",
    "word_مهذب_v02",
    "word_نتعرف_v01",
    "word_وعد_v04",
]

ROUND7_SOURCE_DIRS = [
    DIR_COMPARISON6,
    DIR_COMPARISON5,
    DIR_COMPARISON5_CUT,
    DIR_COMPARISON4,
    DIR_COMPARISON4_CUT,
    DIR_COMPARISON3,
    DIR_COMPARISON3_CUT,
    DIR_COMPARISON2,
    DIR_COMPARISON2_CUT,
    DIR_COMPARISON,
    DIR_CUT,
]

ROUND7_MOVED_TO_CHOSEN: list[dict] = []
ROUND7_ALREADY_CHOSEN: list[str] = []
ROUND7_REMOVED_DUPLICATES: list[dict] = []
ROUND7_MISSING: list[str] = []
ROUND7_CUT_LOG: list[dict] = []

print("=" * 60)
print("MOVE ROUND 7 APPROVED FILES TO chosen/")
print("=" * 60)
print("chosen folder:", DIR_CHOSEN.relative_to(BASE_DIR))

for stem in ROUND7_MOVE_TO_CHOSEN_STEMS:
    filename = f"{stem}.mp3"
    dest = DIR_CHOSEN / filename
    source = None

    if dest.is_file() and dest.stat().st_size > 0:
        ROUND7_ALREADY_CHOSEN.append(filename)
        print(f"already in chosen (not overwritten): {dest.relative_to(BASE_DIR)}")
    else:
        for folder in ROUND7_SOURCE_DIRS:
            candidate = folder / filename
            if candidate.is_file() and candidate.stat().st_size > 0:
                source = candidate
                break

        if source is None:
            ROUND7_MISSING.append(filename)
            print(f"MISSING: {filename}")
            continue

        shutil.move(str(source), str(dest))
        ROUND7_MOVED_TO_CHOSEN.append({
            "file": filename,
            "source": source.parent.relative_to(BASE_DIR),
            "destination": DIR_CHOSEN.relative_to(BASE_DIR),
        })
        print(f"MOVED: {filename}")
        print(f"  source folder: {source.parent.relative_to(BASE_DIR)}")
        print(f"  destination folder: {DIR_CHOSEN.relative_to(BASE_DIR)}")

    if dest.is_file() and dest.stat().st_size > 0:
        for folder in ROUND7_SOURCE_DIRS:
            duplicate = folder / filename
            if duplicate.is_file():
                duplicate.unlink()
                ROUND7_REMOVED_DUPLICATES.append({
                    "file": filename,
                    "removed_from": folder.relative_to(BASE_DIR),
                })
                print(f"REMOVED duplicate: {duplicate.relative_to(BASE_DIR)}")

print("=" * 60)
print("CUT word_الله_v02 TO comparison7/cut/")
print("=" * 60)

allah_src = None
for folder in ROUND7_SOURCE_DIRS:
    candidate = folder / "word_الله_v02.mp3"
    if candidate.is_file() and candidate.stat().st_size > 0:
        allah_src = candidate
        break

if allah_src is None:
    ROUND7_MISSING.append("word_الله_v02.mp3")
    print("MISSING cut source: word_الله_v02.mp3")
else:
    audio = AudioSegment.from_mp3(allah_src)
    original_sec = len(audio) / 1000.0
    cut_audio = audio[:600]
    cut_sec = len(cut_audio) / 1000.0
    out = DIR_COMPARISON7_CUT / "word_الله_v02_cut_0_6.mp3"
    cut_audio.export(out, format="mp3")
    print(f"source: {allah_src.relative_to(BASE_DIR)}")
    print(f"original duration: {original_sec:.3f}s")
    print(f"cut duration: {cut_sec:.3f}s")
    print(f"saved path: {out.relative_to(BASE_DIR)}")
    ROUND7_CUT_LOG.append({
        "source": allah_src,
        "out": out,
        "original_sec": original_sec,
        "cut_sec": cut_sec,
    })

chosen_checks = [DIR_CHOSEN / f"{stem}.mp3" for stem in ROUND7_MOVE_TO_CHOSEN_STEMS]
comparison_duplicates = []
for stem in ROUND7_MOVE_TO_CHOSEN_STEMS:
    filename = f"{stem}.mp3"
    for folder in ROUND7_SOURCE_DIRS:
        duplicate = folder / filename
        if duplicate.is_file():
            comparison_duplicates.append(duplicate)

cut_path = DIR_COMPARISON7_CUT / "word_الله_v02_cut_0_6.mp3"

print("=" * 60)
print("ROUND 7 SUMMARY")
print("=" * 60)
print("files moved to chosen:", len(ROUND7_MOVED_TO_CHOSEN))
for item in ROUND7_MOVED_TO_CHOSEN:
    print(f" - {item['file']} | {item['source']} -> {item['destination']}")
print("already in chosen:", len(ROUND7_ALREADY_CHOSEN))
for filename in ROUND7_ALREADY_CHOSEN:
    print(f" - {filename}")
print("duplicates removed:", len(ROUND7_REMOVED_DUPLICATES))
for item in ROUND7_REMOVED_DUPLICATES:
    print(f" - {item['file']} from {item['removed_from']}")
print("cut file created:", cut_path.is_file())
if cut_path.is_file():
    print(" -", cut_path.relative_to(BASE_DIR))
print("missing files:", len(ROUND7_MISSING))
for item in ROUND7_MISSING:
    print(" -", item)
print("all moved files exist in chosen:", all(path.is_file() for path in chosen_checks))
print("old comparison copies remaining:", len(comparison_duplicates))
if comparison_duplicates:
    for path in comparison_duplicates:
        print(" -", path.relative_to(BASE_DIR))

MOVE ROUND 7 APPROVED FILES TO chosen/
chosen folder: generated_audio_words\edge_tts_words\chosen
MOVED: word_اتفضل_v01.mp3
  source folder: generated_audio_words\edge_tts_words\comparison6
  destination folder: generated_audio_words\edge_tts_words\chosen
MOVED: word_أسمي_v01.mp3
  source folder: generated_audio_words\edge_tts_words\comparison6
  destination folder: generated_audio_words\edge_tts_words\chosen
MOVED: word_أنا_v01.mp3
  source folder: generated_audio_words\edge_tts_words\comparison6
  destination folder: generated_audio_words\edge_tts_words\chosen
MOVED: word_أهلا_v02.mp3
  source folder: generated_audio_words\edge_tts_words\comparison6
  destination folder: generated_audio_words\edge_tts_words\chosen
MOVED: word_بحب_v01.mp3
  source folder: generated_audio_words\edge_tts_words\comparison6
  destination folder: generated_audio_words\edge_tts_words\chosen
MOVED: word_بركاته_v04.mp3
  source folder: generated_audio_words\edge_tts_words\comparison6
  destination folder: gen

## Round 7 Playback

Display playback widgets for selected moved chosen examples and the new `word_الله_v02_cut_0_6.mp3` cut.

In [3]:
ROUND7_PLAYBACK_EXAMPLES = [
    "word_اتفضل_v01.mp3",
    "word_أسمي_v01.mp3",
    "word_أنا_v01.mp3",
    "word_أهلا_v02.mp3",
    "word_نتعرف_v01.mp3",
]

print("Moved chosen examples:")
for filename in ROUND7_PLAYBACK_EXAMPLES:
    path = DIR_CHOSEN / filename
    print(f" - {path.relative_to(BASE_DIR)} | exists: {path.is_file()}")
    if path.is_file():
        display(IPA(filename=str(path), autoplay=False))

print("=" * 60)
print("word_الله_v02_cut_0_6")
print("=" * 60)
allah_cut = DIR_COMPARISON7_CUT / "word_الله_v02_cut_0_6.mp3"
print(f" - {allah_cut.relative_to(BASE_DIR)} | exists: {allah_cut.is_file()}")
if allah_cut.is_file():
    display(IPA(filename=str(allah_cut), autoplay=False))

Moved chosen examples:
 - generated_audio_words\edge_tts_words\chosen\word_اتفضل_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_أسمي_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_أنا_v01.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_أهلا_v02.mp3 | exists: True


 - generated_audio_words\edge_tts_words\chosen\word_نتعرف_v01.mp3 | exists: True


word_الله_v02_cut_0_6
 - generated_audio_words\edge_tts_words\comparison7\cut\word_الله_v02_cut_0_6.mp3 | exists: True


## Move الله Cut to chosen/

Move `word_الله_v02_cut_0_6.mp3` from `comparison7/cut/` into `generated_audio_words/edge_tts_words/chosen/`.

Do not overwrite an existing chosen file; remove the comparison7/cut copy only after the chosen copy exists.

In [2]:
ALLAH_CUT_FILENAME = "word_الله_v02_cut_0_6.mp3"
ALLAH_CUT_SOURCE = DIR_COMPARISON7_CUT / ALLAH_CUT_FILENAME
ALLAH_CUT_DEST = DIR_CHOSEN / ALLAH_CUT_FILENAME

DIR_CHOSEN.mkdir(parents=True, exist_ok=True)

allah_cut_moved = False
allah_cut_missing = False
allah_cut_removed_duplicate = False

print("file:", ALLAH_CUT_FILENAME)
print("source folder:", DIR_COMPARISON7_CUT.relative_to(BASE_DIR))
print("destination folder:", DIR_CHOSEN.relative_to(BASE_DIR))

if ALLAH_CUT_DEST.is_file() and ALLAH_CUT_DEST.stat().st_size > 0:
    print("already in chosen; not overwriting existing chosen file")
    if ALLAH_CUT_SOURCE.is_file():
        ALLAH_CUT_SOURCE.unlink()
        allah_cut_removed_duplicate = True
        print("removed duplicate from source folder:", DIR_COMPARISON7_CUT.relative_to(BASE_DIR))
elif ALLAH_CUT_SOURCE.is_file() and ALLAH_CUT_SOURCE.stat().st_size > 0:
    shutil.move(str(ALLAH_CUT_SOURCE), str(ALLAH_CUT_DEST))
    allah_cut_moved = True
    print("file moved:", ALLAH_CUT_FILENAME)
    print("source folder:", DIR_COMPARISON7_CUT.relative_to(BASE_DIR))
    print("destination folder:", DIR_CHOSEN.relative_to(BASE_DIR))
else:
    allah_cut_missing = True
    print("MISSING file warning:", ALLAH_CUT_FILENAME)

print("chosen exists:", ALLAH_CUT_DEST.is_file())
print("comparison7/cut exists:", ALLAH_CUT_SOURCE.is_file())
print("moved:", allah_cut_moved)
print("duplicate removed:", allah_cut_removed_duplicate)
print("missing:", allah_cut_missing)

file: word_الله_v02_cut_0_6.mp3
source folder: generated_audio_words\edge_tts_words\comparison7\cut
destination folder: generated_audio_words\edge_tts_words\chosen
file moved: word_الله_v02_cut_0_6.mp3
source folder: generated_audio_words\edge_tts_words\comparison7\cut
destination folder: generated_audio_words\edge_tts_words\chosen
chosen exists: True
comparison7/cut exists: False
moved: True
duplicate removed: False
missing: False


## الله Cut Playback

Display the moved chosen `word_الله_v02_cut_0_6.mp3` file.

In [3]:
print("الله cut playback:")
print(f" - {ALLAH_CUT_DEST.relative_to(BASE_DIR)} | exists: {ALLAH_CUT_DEST.is_file()}")
if ALLAH_CUT_DEST.is_file():
    display(IPA(filename=str(ALLAH_CUT_DEST), autoplay=False))

الله cut playback:
 - generated_audio_words\edge_tts_words\chosen\word_الله_v02_cut_0_6.mp3 | exists: True


## Final Chosen Folder

This notebook is for **pronunciation experimentation** only.

After you listen and pick the best variant per word, move (or copy) the winning MP3 files into:

`generated_audio_words/edge_tts_words/chosen/`

Approved number audio should already live under the numbers mapping folders; copies linked above are for reference alongside word picks.